# ScienceDirect 2026 — two-stage contact reason and root cause analysis

Classifies every 2026 ScienceDirect ticket by **contact reason** (13 categories), then by
**root cause** using one dedicated module per contact reason.

**Taxonomy and gold labels** come from `data/ScienceDirect Ops Gold Set of Tickets New -
Alejandro.xlsx`:

| sheet | role |
|---|---|
| `Def CR` | 13 contact reasons + definitions |
| `Def RC` | 56 root causes with short definitions, grouped by contact reason |
| `Lists CR and RC` | allowed CR to RC pairs |
| `Final List` | 163 human-labelled tickets (contact reason + root cause) |

**Tickets** come from `data/sd_collapsed_2026.csv` — 23,344 rows, of which 23,055 are calendar
2026 (Jan–Jul).

## Why two stages

Each contact reason has its own small list of allowed root causes (1–10). Giving the model all
56 at once invites it to pick a root cause that cannot belong to the contact reason it just
assigned. Instead, stage 1 assigns the contact reason, tickets are **split by predicted contact
reason code**, and each subset goes to a module whose taxonomy contains only that contact
reason's root causes. Invalid pairs become impossible by construction.

The pipeline has no way to pass stage 1's answer into a stage 2 prompt, so the routing is done
at the data layer — the same approach as `06_leapspace_2026_two_step.ipynb`.

## Order of operations

Sections 0–6 are **free** and all assert their own results. Sections 7–9 are cheap gates that
must pass before the expensive run. Read the output of section 4 before spending anything.

| section | what | cost |
|---|---|---|
| 0–3 | setup, taxonomy, definition workbook, build | $0 |
| 4 | **Gate 1** — review the exact prompts | $0 |
| 5–6 | preprocess tickets, generate profiles | $0 |
| 7 | **Gate 3** — 30-ticket smoke | ~$0.25 |
| 8 | **Gate 4** — eval on 163 labelled tickets | ~$2 |
| 9 | **Gate 5** — 1,000-record scale probe | ~$8 |
| 10–13 | full run: stage 1, split, stage 2, merge | ~$275, ~8 h |
| 14–16 | eval, feedback, deliverable | ~$2 |

## Two things that will silently cost you money if ignored

1. **Checkpoint resume ignores prompt changes.** The pipeline resumes from
   `runs/checkpoints/<stem>_checkpoint.jsonl` and reports `skip=N todo=0` with zero LLM calls
   whenever every module already has a result. After an interruption that is exactly right;
   after editing the taxonomy it hands back stale answers. Section 6 fingerprints each module's
   prompt **separately** and drops only the affected checkpoint — a global fingerprint would
   turn a one-module fix into a full re-run.
2. **Building overwrites `generated/pipeline_config.json` wholesale.** It is rebuilt from the
   input workbook only, so modules absent from that workbook are dropped. Section 0c backs it up.
   The runtime path is hardcoded (`rca_pipeline/pipeline.py`), and `rca run` has no
   `--config-file`, so redirecting the build elsewhere is not an option — a side config would
   never be read.

## 0. Configuration — edit this cell

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import unicodedata
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# The system Python has pandas built against NumPy 1.x under NumPy 2.x and
# raises "_ARRAY_API not found" on import. Fail now rather than mid-run.
assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} — expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
TICKETS_CSV = BASE / 'data' / 'sd_collapsed_2026.csv'
GOLD_XLSX   = BASE / 'data' / 'ScienceDirect Ops Gold Set of Tickets New - Alejandro.xlsx'
SHEET_DEF_CR   = 'Def CR'
SHEET_DEF_RC   = 'Def RC'
SHEET_LISTS    = 'Lists CR and RC'
SHEET_GOLD     = 'Final List'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'sd_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_sd2026'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'
DATE_COL = 'Chat_Start_Date'

# -- Stage 1 ---------------------------------------------------------------
MOD_CR    = 'sd_contact_reason'
PREFIX_CR = 'SDCR'
STUB_CR   = 'sdcr'

# -- Stage 2: one module per contact reason ------------------------------------
# (contact reason label, slug, module suffix prefix, field stub)
# Labels must match `Def CR` exactly after norm_label(). Order is cosmetic; the
# taxonomy codes come from Def CR row order.
RC_MODULES = [
    ('Access blocked by security/network controls', 'security_block',      'SDSB', 'sdsb'),
    ('Access reports',                              'access_reports',      'SDAR', 'sdar'),
    ('Institutional sign in not working',           'institutional_signin','SDIS', 'sdis'),
    ('Close delete account',                        'close_delete',        'SDCD', 'sdcd'),
    ('Login issues',                                'login',               'SDLI', 'sdli'),
    ('Bulk account issue',                          'bulk_account',        'SDBA', 'sdba'),
    ('Set up teams or departments',                 'teams_departments',   'SDTD', 'sdtd'),
    ('Update email address',                        'update_email',        'SDUE', 'sdue'),
    ('Account configuration for download articles', 'download_config',     'SDDC', 'sddc'),
    ('Accessing content',                           'accessing_content',   'SDAC', 'sdac'),
    ('Other - not access related',                  'other_not_access',    'SDNA', 'sdna'),
]

# Contact reasons with exactly one allowed root cause: assigned in section 13
# with no LLM call. Verified against both taxonomy sheets.
DETERMINISTIC_RC = {
    'Other - access related': 'Other',
    'Not enough info':        'Not enough info',
}

MODULE_PREFIXES = {MOD_CR: PREFIX_CR}
MODULE_PREFIXES.update({f'sd_rc_{slug}': pfx for _, slug, pfx, _ in RC_MODULES})
ALL_MODULES = list(MODULE_PREFIXES)

# -- Preprocessing parameters ---------------------------------------------
# Folded into the stage-1 fingerprint: changing any of them changes the input
# text without changing a prompt, and the checkpoint keys on record content.
SHORT_CHAR_THRESHOLD = 150     # payload chars below which a ticket is 'Not enough info'
TRUNC_CAP            = 16_000  # hard ceiling on any conversation after truncation
TRUNC_HEAD           = 4_000   # kept from the start (the customer's original ask)
TRUNC_TAIL           = 11_900  # kept from the end (where the resolution lives)
TARGET_YEAR          = 2026
# head + tail + the marker itself must stay under the ceiling, hence 11,900 not
# 12,000: a naive 4,000 + 12,000 split plus the marker lands at 16,040.
assert TRUNC_HEAD + TRUNC_TAIL + 64 <= TRUNC_CAP

# -- Outputs ---------------------------------------------------------------
OUT_DIR     = BASE / 'runs' / 'output'
INTER_DIR   = BASE / 'runs' / 'intermediate'
SMOKE_DIR   = BASE / 'runs' / 'smoke_sd_2026'
PROFILE_DIR = BASE / 'profiles'
FEEDBACK_DIR = BASE / 'feedback' / 'sd_2026'
STAGE1_INPUT = BASE / 'data' / 'sd_stage1_input_2026.csv'
FINAL_XLSX   = OUT_DIR / 'sd_2026_two_stage.xlsx'
FINAL_CSV    = OUT_DIR / 'sd_2026_two_stage_full_text.csv'
for _d in (OUT_DIR, INTER_DIR, SMOKE_DIR, PROFILE_DIR, FEEDBACK_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PROFILE_CR = PROFILE_DIR / 'sd_2026_step1_contact_reason.json'

# -- Subprocess environment (UTF-8 matters on Windows) ---------------------
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONUTF8'] = '1'
SUBPROCESS_ENV['PYTHONIOENCODING'] = 'utf-8'

for _p in (TICKETS_CSV, GOLD_XLSX):
    assert _p.exists(), f'Missing: {_p}'
print('Project root :', BASE)
print('Python       :', sys.executable)
print('pandas       :', pd.__version__)
print(f'Modules      : 1 contact reason + {len(RC_MODULES)} root cause '
      f'+ {len(DETERMINISTIC_RC)} deterministic')
print('Paths OK')

## 0b. Shared helpers

`norm_label` is the one function everything else depends on. The gold workbook's labels carry
zero-width spaces (U+200B), non-breaking spaces (U+00A0), leading spaces (`" Content issue"`)
and trailing spaces (`"autorenewal for D2R subscripton "`). These are join keys, so an
unnormalised label silently becomes a different category.

Order matters: zero-width characters are deleted **before** NFKC, because NFKC does not remove
them and a stray U+200B inside a word would survive into the key. The source misspelling
`subscripton` is preserved deliberately — it is the label the gold data uses, and "correcting"
it would break the join.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '​‌‍﻿'))


def norm_label(value) -> str:
    """Normalise a taxonomy/gold label to a stable join key, preserving wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)                    # delete before NFKC
    text = unicodedata.normalize('NFKC', text)            # folds U+00A0 -> ' '
    return re.sub(r'\s+', ' ', text).strip()


def norm_prose(value) -> str:
    """Normalise a definition/description cell. Same as norm_label; kept separate
    because prose is not a join key and may later want different treatment."""
    return norm_label(value)


def norm_cmp(value) -> str:
    """Case-insensitive comparison key, for reconciling sheets and gold labels."""
    return norm_label(value).casefold()


def join_parts(*parts) -> str:
    """Join (heading, text) pairs, skipping any with empty text."""
    return ' '.join(f'{head}: {body}' for head, body in parts if body)


def slugify(text) -> str:
    """Deterministic ASCII slug, used for subset filenames."""
    s = norm_label(text).lower()
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s)).strip('_')


def leaf_labels(module: str) -> dict:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


print('Helpers defined.')

### Self-test for `norm_label`

Every case below is a real string from the gold workbook. If one of these regresses, the
CR to RC join silently splits a category in two.

In [ ]:
_CASES = [
    ('Set up teams or departments​', 'Set up teams or departments'),   # zero-width space
    (' Content issue',                    'Content issue'),                  # leading space
    ('autorenewal for D2R subscripton ',  'autorenewal for D2R subscripton'),# trailing + typo kept
    ('Admin/User\xa0management issue',    'Admin/User management issue'),    # non-breaking space
    ('Browser compatibility issues ​', 'Browser compatibility issues'),
    (None, ''), ('nan', ''), ('   ', ''),
]
for raw, want in _CASES:
    got = norm_label(raw)
    assert got == want, f'norm_label({raw!r}) -> {got!r}, expected {want!r}'
print(f'norm_label: all {len(_CASES)} cases pass.')

## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook
only, so any module not in that workbook is dropped. The config currently holds the three
LeapSpace modules; after section 3 it will hold the twelve ScienceDirect ones.

That is acceptable — the LeapSpace run is finished and its outputs are saved under
`runs/output/leapspace_2026_*` — but the backup makes it reversible. Note this writes to a
**new** directory: reusing `_backup_pre_leapspace2026/` would hit notebook 06's
"never clobber an earlier backup" rule and silently keep a stale copy.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_backed_up = []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *sorted(TAXONOMY_DIR.glob('leapspace_*.json'))]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}')
for _n in _backed_up:
    print(f'  {_n}')

_live = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
print(f'\nModules in the live config right now: {list(_live["modules"])}')

_restore = ('rca build --input '
            'definitions/leapspace_2026_pipeline_definition.xlsx --force-all')
(CONFIG_BACKUP / 'RESTORE.txt').write_text(
    'To put the LeapSpace modules back in generated/pipeline_config.json:\n\n'
    f'    {_restore}\n\n'
    'Its outputs are already saved under runs/output/leapspace_2026_*.\n',
    encoding='utf-8')
print(f'\nTo restore LeapSpace later:\n    {_restore}')

## 1. Read the taxonomy out of the gold workbook

Three sheets, two of which describe the same CR to RC structure and disagree in exactly one
place.

**`Def RC` group column is sparse** — the contact reason appears once per group and is blank on
the rows beneath. It must be forward-filled, and `norm_label` returns `''` rather than `NaN`, so
a bare `.ffill()` would be a no-op and every root cause would land under the first contact
reason. Hence the `.replace('', pd.NA)` before `.ffill()`.

**The one disagreement:** `Request for information` sits under `Accessing content` in `Def RC`
but under `Other - not access related` in `Lists CR and RC`. The human-labelled data puts it
under `Other - not access related`, so **`Lists CR and RC` wins for parentage** and `Def RC`
supplies the description text. This is flagged in the deliverable for the taxonomy owner.

In [ ]:
# ---- Def CR: 13 contact reasons, one definition each ----------------------
def_cr = pd.read_excel(GOLD_XLSX, sheet_name=SHEET_DEF_CR, dtype=str)
def_cr.columns = [norm_label(c) for c in def_cr.columns]
CR_DEFS = {}                                    # label -> definition, in sheet order
for _, row in def_cr.iterrows():
    label = norm_label(row['Contact reason'])
    if label and label.casefold() != 'contact reason':
        CR_DEFS[label] = norm_prose(row['Definition'])
CR_ORDER = list(CR_DEFS)

# ---- Def RC: root cause -> short definition, grouped by contact reason ----
def_rc = pd.read_excel(GOLD_XLSX, sheet_name=SHEET_DEF_RC, dtype=str)
def_rc.columns = [norm_label(c) for c in def_rc.columns]
def_rc['Contact Reason'] = (def_rc['Contact Reason'].map(norm_label)
                            .replace('', pd.NA).ffill())      # required, see above
def_rc['Root Cause'] = def_rc['Root Cause'].map(norm_label)
def_rc['Short definition'] = def_rc['Short definition'].map(norm_prose)
def_rc = def_rc[def_rc['Root Cause'] != ''].copy()

# Keyed case-insensitively: the two sheets differ in the casing of
# 'Update account bulk ids', which is a spelling variant of the same pair and
# must not be mistaken for a parentage disagreement.
RC_DEFN_BY_PAIR = {(norm_cmp(r['Contact Reason']), norm_cmp(r['Root Cause'])):
                   r['Short definition'] for _, r in def_rc.iterrows()}
RC_DEFN_BY_NAME = {}                            # fallback for a genuinely relocated pair
RC_PARENT_IN_DEFRC = {}                         # rc -> the CR Def RC files it under
for _, r in def_rc.iterrows():
    RC_DEFN_BY_NAME.setdefault(norm_cmp(r['Root Cause']), r['Short definition'])
    RC_PARENT_IN_DEFRC.setdefault(norm_cmp(r['Root Cause']), r['Contact Reason'])

# ---- Lists CR and RC: authoritative CR -> [allowed root causes] ----------
lists_df = pd.read_excel(GOLD_XLSX, sheet_name=SHEET_LISTS, dtype=str)
CR_TO_RC = {}
for col in lists_df.columns[1:]:                # col 0 is the row-label gutter
    cr = norm_label(col)
    if not cr:
        continue
    seen, allowed = set(), []
    for v in lists_df[col]:
        rc = norm_label(v)
        if rc and rc.casefold() != 'root cause' and norm_cmp(rc) not in seen:
            seen.add(norm_cmp(rc))
            allowed.append(rc)
    CR_TO_RC[cr] = allowed

# ---- Integrity ----------------------------------------------------------
assert len(CR_DEFS) == 13, f'expected 13 contact reasons in Def CR, got {len(CR_DEFS)}'
assert set(CR_TO_RC) == set(CR_DEFS), (
    f'sheet mismatch\n  only in Lists : {sorted(set(CR_TO_RC) - set(CR_DEFS))}'
    f'\n  only in Def CR: {sorted(set(CR_DEFS) - set(CR_TO_RC))}')
assert all(d for d in CR_DEFS.values()), 'a contact reason has no definition'
_n_pairs = sum(len(v) for v in CR_TO_RC.values())
assert _n_pairs == 56, f'expected 56 CR/RC pairs, got {_n_pairs}'

print(f'{len(CR_DEFS)} contact reasons, {_n_pairs} allowed CR/RC pairs\n')
for cr in CR_ORDER:
    flag = '  [deterministic]' if cr in DETERMINISTIC_RC else ''
    print(f'  {len(CR_TO_RC[cr]):>2} RC   {cr}{flag}')

# ---- Report the Def RC vs Lists disagreement explicitly -----------------
_pairs_lists = {(norm_cmp(cr), norm_cmp(rc)) for cr, v in CR_TO_RC.items() for rc in v}
_pairs_defrc = set(RC_DEFN_BY_PAIR)
print('\nDef RC vs Lists CR and RC (compared case-insensitively):')
if _pairs_defrc == _pairs_lists:
    print('  the two sheets agree exactly')
for cr, rc in sorted(_pairs_defrc - _pairs_lists):
    print(f'  only in Def RC : {rc!r} under {cr!r}')
for cr, rc in sorted(_pairs_lists - _pairs_defrc):
    print(f'  only in Lists  : {rc!r} under {cr!r}   <- Lists wins (matches gold labels)')


def rc_description(cr: str, rc: str) -> str:
    """Short definition for one allowed pair.

    Falls back by root-cause name alone when Lists CR and RC files the pair under
    a different contact reason than Def RC does, so the relocated pair still
    reaches the prompt with its definition intact.
    """
    return (RC_DEFN_BY_PAIR.get((norm_cmp(cr), norm_cmp(rc)))
            or RC_DEFN_BY_NAME.get(norm_cmp(rc), ''))


# ---- Every allowed pair must resolve to a description --------------------
TAXONOMY_NOTES = []
_missing = []
for cr, rcs in CR_TO_RC.items():
    for rc in rcs:
        if RC_DEFN_BY_PAIR.get((norm_cmp(cr), norm_cmp(rc))):
            continue
        if RC_DEFN_BY_NAME.get(norm_cmp(rc)):
            TAXONOMY_NOTES.append(
                f'{rc!r} is listed under {cr!r} by Lists CR and RC but under '
                f'{RC_PARENT_IN_DEFRC[norm_cmp(rc)]!r} by Def RC; kept under '
                f'{cr!r} and given the Def RC definition')
            continue
        _missing.append((cr, rc))
assert not _missing, f'allowed pairs with no Def RC description: {_missing}'
print()
for n in TAXONOMY_NOTES:
    print(f'  note: {n}')


print('\nTaxonomy loaded and reconciled.')

## 2. Write the pipeline definition workbook

`rca build` consumes three sheets — `modules`, `fields`, `taxonomy` — and **assigns
taxonomy codes by row order**, so codes are never written by hand. `Def CR` row order therefore
fixes the contact reason codes (`SDCR1.1` … `SDCR13.1`).

Every taxonomy is two-level because `rca build` unconditionally appends *"Always use
LEAF-level codes"* to the generated prompt, so an L1-only code would contradict its own
instructions. For the contact reason module each category is an L1 group with a single L2 leaf
repeating the label, and the guidance sits on the **leaf only** — `taxonomy_to_rich_prompt`
prints both levels, so text on both would duplicate it.

**Definitions are copied verbatim.** `Def CR` has one prose `Definition` per contact reason, not
LeapSpace's three separate columns, and the connector varies across the thirteen ("and not for",
"and not when", "and only when", one with no negative clause at all). Splitting the sentence
into "Use when" / "Do not use" sections would silently produce empty sections, so it is emitted
whole.

Two rules are **authored**, not from the workbook — both are marked in the `notes` column.

In [ ]:
# ══ Authored additions ═══════════════════════════════════════════════════
# Not present in the gold workbook. Everything else below is verbatim.

# 1. Stage-1 precedence. Read off the "and not for" clauses in Def CR, which
#    describe the boundaries pairwise but never state a decision order.
CR_PRECEDENCE = (
    'Decide what actually blocked the customer before what they were talking about. '
    'If an explicit security or network control denied access — access denied, IP blocked, '
    'Cloudflare, firewall, suspected crawling — classify the ticket as '
    "'Access blocked by security/network controls' even when content or signing in is also "
    'mentioned. If the obstacle is a failed sign-in, choose between '
    "'Institutional sign in not working' (SSO, SAML, Shibboleth, institution not recognised) "
    "and 'Login issues' (an individual account, password, activation or verification link) by "
    "asking who owns the credentials. Use 'Accessing content' only when signing in worked and "
    'the failure is entitlement, coverage or a paywall. Prefer a specific category over either '
    "'Other' category, and use 'Not enough info' only when the ticket genuinely does not say "
    'enough to tell. '
    # Where money is involved the two sheets pull in opposite directions:
    # 'Accessing content' lists "purchase not recognised" among its signals, while
    # 'Other - not access related' lists "invoices, sales" and "content corrections".
    # This separates the transaction from the entitlement, which is faithful to both.
    'When money is involved, separate the transaction from the entitlement. If the complaint is '
    'about the payment or the order itself — the payment failed or would not go through, a double '
    'charge, a missing receipt or confirmation, a refund, an invoice or a price — classify it as '
    "'Other - not access related', whose own signals name invoices and sales. Send complaints "
    'about the content itself in the same direction — poor or wrong images, typographical errors, '
    "or requests for corrections. Reserve 'Accessing content' for the case where the purchase or "
    'subscription is not in dispute but the content still will not open: purchase not recognised '
    'means a completed purchase the system fails to honour, not a purchase that failed.'
)

# 2. Stage-2 evidence threshold. Twelve of the thirteen root-cause groups contain
#    both 'Not enough info' and 'Other', so without this the model over-commits to
#    a specific cause on thin evidence rather than using them.
RC_EVIDENCE_RULE = (
    'A specific root cause applies only when the signals named in its own definition are '
    'actually present. Do not infer a specific technical cause from a symptom alone. Where the '
    'ticket shows only generic troubleshooting and never establishes what was actually wrong, '
    "use the Not enough info code rather than guessing. Use 'Other' only when the ticket does "
    'establish a cause but none of the listed root causes describes it.'
)

JUST_INSTR = (
    'In one or two sentences, explain why this {kind} applies and why the closest alternative '
    'does not. Refer to what the ticket actually shows. Do not use apostrophes or quotation '
    'marks in your answer.'
)
EV_INSTR = (
    'Copy verbatim the single sentence or phrase from the transcript that most strongly '
    'supports this {kind}. Do not paraphrase, infer, or add words that are not in the '
    'transcript. If no direct evidence exists, return an empty string.'
)


def build_modules_sheet() -> pd.DataFrame:
    rows = [{
        'module_name': MOD_CR,
        'taxonomy_prefix': PREFIX_CR,
        'taxonomy_file': f'{MOD_CR}.json',
        'reorder_group': 'A', 'reorder_order': 1, 'enabled': True,
        'description': (
            f'Classify the reason someone contacted ScienceDirect support into exactly one of '
            f'{len(CR_DEFS)} categories. Each category carries a Definition saying when to use '
            f'it and what not to use it for. Assign the single best-fitting LEAF code '
            f'(e.g. {PREFIX_CR}5.1). ' + CR_PRECEDENCE),
        'notes': 'Stage 1 of 2. Runs on every ticket.',
    }]
    for order, (cr, slug, pfx, _stub) in enumerate(RC_MODULES, start=2):
        rows.append({
            'module_name': f'sd_rc_{slug}',
            'taxonomy_prefix': pfx,
            'taxonomy_file': f'sd_rc_{slug}.json',
            'reorder_group': 'B', 'reorder_order': order, 'enabled': True,
            'description': (
                f'Identify the root cause for ScienceDirect tickets already classified as '
                f"'{cr}'. Each root cause carries a Definition — use these to separate root "
                f'causes that look similar. Assign the single best-fitting LEAF code '
                f'(e.g. {pfx}1.1).'),
            'notes': f'Stage 2. Runs only on the {slug} subset ({len(CR_TO_RC[cr])} root causes).',
        })
    return pd.DataFrame(rows)


def build_fields_sheet() -> pd.DataFrame:
    """Three fields per module: code, justification, verbatim evidence.

    Field stubs deliberately avoid the `contact_reason_` / `root_cause_` prefixes
    in utils._STRUCTURAL_PREFIXES, which drop_enrichment_cols() would strip from
    any re-run input.
    """
    specs = [(MOD_CR, STUB_CR, PREFIX_CR, 'contact reason',
              f'Assign the single leaf-level code (e.g. {PREFIX_CR}5.1) for the contact reason '
              f'that best matches this ticket. Read each category Definition and match on what '
              f'the ticket shows, not on surface keywords. Only use codes from the taxonomy.')]
    for cr, slug, pfx, stub in RC_MODULES:
        specs.append((f'sd_rc_{slug}', stub, pfx, 'root cause',
                      f'Assign the single leaf-level code (e.g. {pfx}1.1) for the root cause '
                      f'that best matches this ticket. Match on each root cause Definition, '
                      f'not on surface keywords. Only use codes from the taxonomy. '
                      + RC_EVIDENCE_RULE))
    rows = []
    for module, stub, _pfx, kind, code_instr in specs:
        rows.append({'module_name': module, 'field_name': f'{stub}_code',
                     'field_type': 'code', 'description': 'Assigned leaf code',
                     'prompt_instruction': code_instr, 'in_output': True,
                     'explode_list': False, 'validate_enum': True, 'notes': ''})
        rows.append({'module_name': module, 'field_name': f'{stub}_justification',
                     'field_type': 'str',
                     'description': 'Why this code was chosen over the nearest alternative',
                     'prompt_instruction': JUST_INSTR.format(kind=kind), 'in_output': True,
                     'explode_list': False, 'validate_enum': False, 'notes': ''})
        rows.append({'module_name': module, 'field_name': f'{stub}_evidence',
                     'field_type': 'str',
                     'description': 'Verbatim supporting quote from the transcript',
                     'prompt_instruction': EV_INSTR.format(kind=kind), 'in_output': True,
                     'explode_list': False, 'validate_enum': False, 'notes': ''})
    return pd.DataFrame(rows)


def build_taxonomy_sheet() -> pd.DataFrame:
    rows = []
    # -- stage 1: one L1 group + one L2 leaf per contact reason, in Def CR order
    for cr in CR_ORDER:
        rows.append({'module_name': MOD_CR, 'level': 'L1', 'label': cr,
                     'description': '', 'notes': ''})
        rows.append({'module_name': MOD_CR, 'level': 'L2', 'label': cr,
                     'description': join_parts(('Definition', CR_DEFS[cr])),
                     'notes': ''})
    # -- stage 2: one module per contact reason, gated at L1
    for cr, slug, _pfx, _stub in RC_MODULES:
        module = f'sd_rc_{slug}'
        rows.append({
            'module_name': module, 'level': 'L1', 'label': cr,
            'description': (
                f"Every ticket in this task has already been classified as '{cr}'. Choose the "
                f'root cause from this group only. That contact reason means: {CR_DEFS[cr]}'),
            'notes': 'gating context, authored',
        })
        for rc in CR_TO_RC[cr]:
            rows.append({'module_name': module, 'level': 'L2', 'label': rc,
                         'description': join_parts(('Definition', rc_description(cr, rc))),
                         'notes': ''})
    return pd.DataFrame(rows, columns=['module_name', 'level', 'label',
                                       'description', 'notes'])


modules_df  = build_modules_sheet()
fields_df   = build_fields_sheet()
taxonomy_df = build_taxonomy_sheet()

# ---- Shape checks before anything downstream depends on it ---------------
expected = {MOD_CR: (len(CR_ORDER), len(CR_ORDER))}
for cr, slug, _pfx, _stub in RC_MODULES:
    expected[f'sd_rc_{slug}'] = (1, len(CR_TO_RC[cr]))

for module, (want_l1, want_l2) in expected.items():
    sub = taxonomy_df[taxonomy_df['module_name'] == module]
    got = (int((sub['level'] == 'L1').sum()), int((sub['level'] == 'L2').sum()))
    assert got == (want_l1, want_l2), f'{module}: expected {(want_l1, want_l2)} L1/L2, got {got}'

blank = taxonomy_df[(taxonomy_df['level'] == 'L2') & (taxonomy_df['description'] == '')]
assert blank.empty, f'leaf rows with no guidance:\n{blank[["module_name", "label"]]}'

# _clean_df drops rows whose FIRST column starts with L1/L2/level/module_name/Green/Edit.
# The first column is module_name and every value starts with 'sd_', so nothing is
# dropped — but never reorder the columns to put `level` first.
assert taxonomy_df.columns[0] == 'module_name'
assert taxonomy_df['module_name'].str.startswith('sd_').all()

DEFN_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as w:
    modules_df.to_excel(w, sheet_name='modules', index=False)
    fields_df.to_excel(w, sheet_name='fields', index=False)
    taxonomy_df.to_excel(w, sheet_name='taxonomy', index=False)

print(f'Wrote {DEFN_XLSX.relative_to(BASE)}')
print(f'  {len(modules_df)} modules, {len(fields_df)} fields, '
      f'{len(taxonomy_df)} taxonomy rows')
print(f'  {int((taxonomy_df["level"] == "L2").sum())} leaves total '
      f'({len(CR_ORDER)} contact reasons + '
      f'{int((taxonomy_df["level"] == "L2").sum()) - len(CR_ORDER)} root causes)\n')
for module, (l1, l2) in expected.items():
    print(f'  {module:<28} {MODULE_PREFIXES[module]:<5} {l1:>2} L1 / {l2:>2} L2')

## 3. Build the pipeline artefacts

Dry run first — it reports what would be written without touching anything.

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--dry-run'], 'build_pipeline --dry-run')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', *ALL_MODULES], 'build_pipeline')

# ---- Verify the built shape ---------------------------------------------
cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
assert set(cfg['modules']) == set(ALL_MODULES), (
    f'config modules mismatch\n  missing: {set(ALL_MODULES) - set(cfg["modules"])}'
    f'\n  extra  : {set(cfg["modules"]) - set(ALL_MODULES)}')

for module, (_l1, want_leaves) in expected.items():
    assert (TAXONOMY_DIR / f'{module}.json').exists(), f'missing taxonomy for {module}'
    assert (MODULES_DIR / f'{module}.py').exists(), f'missing {module}.py'
    got = len(leaf_labels(module))
    assert got == want_leaves, f'{module}: expected {want_leaves} leaves, got {got}'
    assert cfg['modules'][module]['taxonomy_prefix'] == MODULE_PREFIXES[module]

# Field stubs must not collide with the prefixes drop_enrichment_cols() strips,
# or results would vanish from the output on a re-run.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
for stub in [STUB_CR] + [s for *_r, s in RC_MODULES]:
    for bad in _STRUCTURAL_PREFIXES:
        assert not f'{stub}_'.startswith(bad), f'field stub {stub}_ collides with {bad!r}'

# Prefixes are parsed with a greedy ^[A-Z]+, so none may prefix another.
_pfx = sorted(MODULE_PREFIXES.values())
for a in _pfx:
    for b in _pfx:
        assert a == b or not b.startswith(a), f'prefix {a!r} is a prefix of {b!r}'

CR_LABELS = leaf_labels(MOD_CR)                 # code -> contact reason label
LABEL_TO_CR_CODE = {v: k for k, v in CR_LABELS.items()}
assert len(LABEL_TO_CR_CODE) == len(CR_LABELS), 'duplicate contact reason labels'

print(f'\nBuild verified: {len(ALL_MODULES)} modules, '
      f'{sum(len(leaf_labels(m)) for m in ALL_MODULES)} leaves total.\n')
print('Contact reason codes:')
for code_, label in CR_LABELS.items():
    n_rc = len(CR_TO_RC[label])
    tag = '(deterministic)' if label in DETERMINISTIC_RC else f'{n_rc} root causes'
    print(f'  {code_:<9} {label:<45} {tag}')

## 4. Gate 1 — review the exact prompts *(free, read this output)*

Renders every system prompt exactly as the model will receive it, then asserts that every
definition from `Def CR` and `Def RC` survives into its module's prompt **verbatim**. A silently
dropped definition or a mis-wired prefix is invisible at every later stage and shows up only as
poor accuracy, so this is the cheapest place to catch it.

`importlib.reload` matters: after a rebuild in section 3 the old module object is still in
`sys.modules`, and without the reload the prompts shown here would be the previous build's.

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
prompts = {}
for module in ALL_MODULES:
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    name = f'generated.modules.{module}'
    mod = importlib.reload(sys.modules[name]) if name in sys.modules \
        else importlib.import_module(name)
    prompts[module] = mod._build_system_prompt(
        taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))

# ---- Verbatim guidance check -------------------------------------------
checked = 0
for cr, defn in CR_DEFS.items():
    assert cr in prompts[MOD_CR], f'contact reason label missing from prompt: {cr!r}'
    assert defn in prompts[MOD_CR], f'contact reason definition missing: {defn[:70]!r}'
    checked += 2
for cr, slug, _pfx, _stub in RC_MODULES:
    p = prompts[f'sd_rc_{slug}']
    for rc in CR_TO_RC[cr]:
        assert rc in p, f'root cause label missing from sd_rc_{slug}: {rc!r}'
        defn = rc_description(cr, rc)
        assert defn in p, f'root cause definition missing from sd_rc_{slug}: {defn[:70]!r}'
        checked += 2
assert CR_PRECEDENCE in prompts[MOD_CR], 'authored precedence rule missing from stage 1'
for _cr, slug, _pfx, _stub in RC_MODULES:
    assert RC_EVIDENCE_RULE in prompts[f'sd_rc_{slug}'], f'evidence rule missing: {slug}'
print(f'Verbatim guidance check: all {checked} taxonomy strings present in the prompts.\n')

# ---- Per-module fingerprints -------------------------------------------
# Stored beside each checkpoint so a prompt edit invalidates only the affected
# stage. A single global hash would force a full ~$275 re-run for a one-module fix.
PROMPT_FINGERPRINTS = {
    m: hashlib.sha256(p.encode('utf-8')).hexdigest()[:16] for m, p in prompts.items()}

# Stage 1 also depends on how the input text was cut, and the checkpoint keys on
# record content — so fold the preprocessing parameters into its fingerprint.
_preproc = f'{SHORT_CHAR_THRESHOLD}|{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}|{TARGET_YEAR}'
PROMPT_FINGERPRINTS[MOD_CR] = hashlib.sha256(
    (prompts[MOD_CR] + _preproc).encode('utf-8')).hexdigest()[:16]

print(f'{"module":<28} {"chars":>7} {"~tokens":>8}  fingerprint')
for module in ALL_MODULES:
    p = prompts[module]
    print(f'{module:<28} {len(p):>7,} {len(p) // 4:>8,}  {PROMPT_FINGERPRINTS[module]}')

In [ ]:
print('=' * 78)
print('FULL STAGE 1 SYSTEM PROMPT')
print('=' * 78)
print(prompts[MOD_CR])

In [ ]:
# Every stage-2 prompt, in full. Long but this is the one chance to read them
# before they cost money.
for _cr, slug, _pfx, _stub in RC_MODULES:
    print('=' * 78)
    print(f'sd_rc_{slug}')
    print('=' * 78)
    print(prompts[f'sd_rc_{slug}'])
    print()

## 5. Gate 2 — preprocess the tickets *(free)*

Three transformations, each asserted:

**Filter to calendar 2026.** The file spans 2023-10 to 2026-07 but is overwhelmingly 2026;
the 289 earlier rows are written out rather than dropped silently.

**Short-ticket rule.** ~2,250 rows carry no information — `"Customer: ."`, `"Customer: no"`,
`"Customer: Ghost Call"`, `"Customer: Call already logged on another Incident"`. Length is
measured on the *payload*, with the `Customer:` / `Agent Notes:` markers stripped, so a ticket
padded past the threshold by its markers alone is still caught. These get `Not enough info`
by rule with no LLM call, and an assert confirms none is a labelled gold ticket — otherwise the
rule would quietly change the eval denominator.

**Truncation to a 16,000-char ceiling, head 4,000 + tail 11,900.** `make_batches` never
truncates: it only *flushes* batches, so a single oversized row is appended and sent whole — the
longest conversation here is 247,499 chars (~62K tokens). The tail is weighted because the
resolution lives at the end of a transcript, but the customer's original ask is in the first few
hundred characters and stage 1 needs it. In the 247K record the head is quoted email boilerplate
while the tail holds the entire resolution. The `[...TRUNCATED...]` marker is visible so the
model does not read a splice as continuous.

The tail is 11,900 rather than 12,000 because the marker counts toward the ceiling: a naive
4,000 + 12,000 split lands at 16,040 and breaks the cap it was supposed to enforce.

This truncates only ~234 of 23,055 rows while keeping ~93% of all text. Its real purpose is to
**bound the worst-case batch**: with `batch_size 5` and `max_chars 40000`, no request can exceed
~20K input tokens whatever mix of rows it draws.

In [ ]:
raw = pd.read_csv(TICKETS_CSV, dtype=str, low_memory=False)
raw[ID_COL] = raw[ID_COL].astype(str).str.strip()
print(f'{TICKETS_CSV.name}: {len(raw):,} rows x {len(raw.columns)} cols')
assert raw[ID_COL].is_unique, 'duplicate INCIDENT_REF in the source file'

# ---- 1. Filter to the target year --------------------------------------
# The parsed date becomes a column *before* filtering. Carrying it as a separate
# Series and using it after .reset_index() would realign on the old index and
# silently drop rows from any later .assign()/groupby.
raw['parsed_date'] = pd.to_datetime(raw[DATE_COL], errors='coerce')
assert raw['parsed_date'].isna().sum() == 0, 'unparseable dates in Chat_Start_Date'

in_year = raw['parsed_date'].dt.year == TARGET_YEAR
excluded = raw[~in_year].copy()
excluded.to_csv(INTER_DIR / 'sd_2026_excluded_pre2026.csv', index=False)
tickets = raw[in_year].copy().reset_index(drop=True)

print(f'\nCalendar {TARGET_YEAR}: {len(tickets):,} kept, {len(excluded)} excluded '
      f'({excluded["parsed_date"].min().date()} .. '
      f'{excluded["parsed_date"].max().date()})')
_by_month = tickets['parsed_date'].dt.to_period('M').value_counts().sort_index()
print(_by_month.to_string())
assert _by_month.sum() == len(tickets), 'month counts do not sum to the kept rows'

# ---- 2. Short-ticket rule ----------------------------------------------
_MARKER_RE = re.compile(r'(?mi)^\s*(customer|agent)(\s+notes)?\s*:', re.MULTILINE)


def payload_len(text) -> int:
    """Conversation length ignoring the speaker markers themselves."""
    return len(_MARKER_RE.sub('', str(text)).strip())


tickets['conversation_chars_original'] = tickets[TEXT_COL].astype(str).str.len()
tickets['payload_chars'] = tickets[TEXT_COL].map(payload_len)
is_short = tickets['payload_chars'] < SHORT_CHAR_THRESHOLD

rule_assigned = tickets[is_short].copy()
rule_assigned['classification_source'] = 'rule:short_conversation'
rule_assigned.to_csv(INTER_DIR / 'sd_2026_rule_assigned.csv', index=False)
to_classify = tickets[~is_short].copy().reset_index(drop=True)

print(f'\nShort-ticket rule (payload < {SHORT_CHAR_THRESHOLD} chars): '
      f'{len(rule_assigned):,} rows set aside, {len(to_classify):,} to classify')
print('  most common content-free values:')
print(rule_assigned[TEXT_COL].str.slice(0, 70).value_counts().head(6).to_string())

# ---- 3. Truncate long conversations ------------------------------------
TRUNC_MARKER = '\n\n[...TRUNCATED {n:,} CHARS FROM MIDDLE...]\n\n'


def truncate_middle(text) -> str:
    s = str(text)
    if len(s) <= TRUNC_CAP:
        return s
    cut = len(s) - TRUNC_HEAD - TRUNC_TAIL
    return s[:TRUNC_HEAD] + TRUNC_MARKER.format(n=cut) + s[-TRUNC_TAIL:]


was_long = to_classify['conversation_chars_original'] > TRUNC_CAP
to_classify['was_truncated'] = was_long
to_classify[TEXT_COL] = to_classify[TEXT_COL].map(truncate_middle)

trunc_log = to_classify.loc[was_long, [ID_COL, 'conversation_chars_original']].copy()
trunc_log['kept_chars'] = to_classify.loc[was_long, TEXT_COL].str.len()
trunc_log.to_csv(INTER_DIR / 'sd_2026_truncation_log.csv', index=False)
print(f'\nTruncated {int(was_long.sum())} of {len(to_classify):,} conversations to '
      f'{TRUNC_CAP:,} chars (head {TRUNC_HEAD:,} + tail {TRUNC_TAIL:,})')
_kept = to_classify[TEXT_COL].str.len().sum()
print(f'  text retained: {_kept:,} of '
      f'{to_classify["conversation_chars_original"].sum():,} chars '
      f'({_kept / to_classify["conversation_chars_original"].sum() * 100:.1f}%)')

# ---- The rule must not touch the eval set ------------------------------
_gold_probe = pd.read_excel(GOLD_XLSX, sheet_name=SHEET_GOLD, dtype=str)
_gold_ids = set(_gold_probe.loc[_gold_probe['Root cause (select)'].notna(),
                               'Unique Identifier Ticket'].map(norm_label))
_hit = _gold_ids & set(rule_assigned[ID_COL])
assert not _hit, (f'{len(_hit)} labelled gold ticket(s) were caught by the short-ticket '
                  f'rule and would silently leave the eval: {sorted(_hit)[:5]}')
print(f'\nNone of the {len(_gold_ids)} labelled gold tickets was caught by the rule.')

# ---- Write the stage-1 input -------------------------------------------
STAGE1_COLS = [ID_COL, 'Incident_ID', 'Channel', DATE_COL, TEXT_COL]
to_classify[STAGE1_COLS].to_csv(STAGE1_INPUT, index=False)

assert len(tickets) == len(to_classify) + len(rule_assigned)
assert to_classify[TEXT_COL].str.len().max() <= TRUNC_CAP, 'a conversation exceeds the cap'
assert to_classify[ID_COL].is_unique
print(f'\n{len(tickets):,} = {len(to_classify):,} to classify + '
      f'{len(rule_assigned):,} rule-assigned   OK')
print(f'Wrote {STAGE1_INPUT.relative_to(BASE)}')

## 6. Generate the profiles and arm the fingerprint guard

Twelve profiles that differ in four strings each are generated from one table rather than
hand-written. The reason is specific: `rca run` applies a profile key only when the
matching CLI arg is still at its parser default, via a `hasattr` guard — so a **key it does not
recognise is silently ignored**. Writing `"max_char"` for `"max_chars"` would not error, it
would quietly use the 60,000 default. Every emitted key is therefore validated against the real
argparse destinations first.

Tuning values are inherited from the LeapSpace run, which completed 1,505 tickets without a
retry storm: `batch_size 5`, `max_chars 40000`, `concurrency 2`, `retries 5`, `max_tokens 8192`.

In [ ]:
# ---- Which keys does rca run actually accept? -------------------
# Read the flags off --help rather than regex-scanning the source: it is the
# same list argparse will really use, and it doubles as a smoke test that
# rca run imports cleanly under this interpreter.
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests, \
    f'could not parse rca run flags (got {sorted(_dests)})'

TUNING = {
    'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4,
    'retries': 5, 'max_tokens': 8192, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
}


def write_profile(path: Path, *, comment, input_rel, output_rel, modules) -> Path:
    spec = {'_comment': comment, 'input': input_rel, 'output': output_rel,
            'taxonomy_dir': 'generated/taxonomies',
            'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': modules, **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


write_profile(
    PROFILE_CR,
    comment=(f'Stage 1 of 2 - classify every {TARGET_YEAR} ScienceDirect ticket into one of '
             f'{len(CR_ORDER)} contact reasons. Taxonomy from the Alejandro gold workbook via '
             f'{DEFN_XLSX.name}. Input is written by notebooks/07_sd_2026_two_stage.ipynb '
             f'(2026-filtered, short tickets removed, conversations capped at '
             f'{TRUNC_CAP} chars) - not the raw ticket file.'),
    input_rel=STAGE1_INPUT.relative_to(BASE).as_posix(),
    output_rel=(OUT_DIR / 'sd_2026_step1_contact_reason.xlsx').relative_to(BASE).as_posix(),
    modules=[MOD_CR])

# ---- One stage-2 profile per contact reason ----------------------------
# ROUTES is the single source of truth for stage 2, keyed by contact reason CODE.
# Routing on the label would be fragile: two contain a zero-width or non-breaking
# space, so a string comparison can silently yield an empty subset. Codes are ASCII.
ROUTES = {}
for cr, slug, pfx, stub in RC_MODULES:
    module = f'sd_rc_{slug}'
    profile = PROFILE_DIR / f'sd_2026_step2_{slug}.json'
    write_profile(
        profile,
        comment=(f"Stage 2 of 2 - root cause for tickets stage 1 classified as '{cr}' "
                 f'({LABEL_TO_CR_CODE[cr]}). Input is the subset CSV written by '
                 f'notebooks/07_sd_2026_two_stage.ipynb, not the full ticket file.'),
        input_rel=(INTER_DIR / f'sd_2026_subset_{slug}.csv').relative_to(BASE).as_posix(),
        output_rel=(OUT_DIR / f'sd_2026_step2_{slug}.xlsx').relative_to(BASE).as_posix(),
        modules=[module])
    ROUTES[LABEL_TO_CR_CODE[cr]] = {
        'cr_label': cr, 'slug': slug, 'module': module, 'prefix': pfx,
        'stub': stub, 'profile': profile, 'kind': 'llm',
        'subset_csv': INTER_DIR / f'sd_2026_subset_{slug}.csv',
    }
for cr, rc in DETERMINISTIC_RC.items():
    ROUTES[LABEL_TO_CR_CODE[cr]] = {
        'cr_label': cr, 'slug': slugify(cr), 'module': '', 'prefix': '',
        'stub': '', 'profile': None, 'kind': 'deterministic', 'root_cause': rc,
    }

assert set(ROUTES) == set(CR_LABELS), (
    f'every contact reason code needs a route\n  missing: {set(CR_LABELS) - set(ROUTES)}')
print(f'Wrote {1 + len(RC_MODULES)} profiles; {len(ROUTES)} contact reason codes routed '
      f'({sum(r["kind"] == "llm" for r in ROUTES.values())} to an LLM module, '
      f'{sum(r["kind"] == "deterministic" for r in ROUTES.values())} deterministic).\n')
for code_ in sorted(ROUTES, key=lambda c: (len(c), c)):
    r = ROUTES[code_]
    dest = r['module'] if r['kind'] == 'llm' else f'(rule) -> {r["root_cause"]!r}'
    print(f'  {code_:<9} {r["cr_label"]:<45} {dest}')

In [ ]:
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list:
    """The checkpoint artefacts rca run keeps for a given input file."""
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    """Read a {module: hash} fingerprint file, tolerating older formats.

    Notebook 06 wrote a single bare 16-char hash here, which is not valid JSON.
    Treat anything unparseable, or a plain string, as 'no fingerprint' so the
    checkpoint is discarded rather than crashing the run.
    """
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, module: str) -> None:
    """Drop this input's checkpoint if it was produced by a different prompt.

    The pipeline resumes from runs/checkpoints/<stem>_checkpoint.jsonl whenever
    every module already has a result for a record - it logs `skip=N todo=0` and
    makes no LLM calls. That is what you want after an interruption and exactly
    what you do not want after editing the taxonomy.

    Keyed per module, so editing one root-cause prompt invalidates one subset's
    checkpoint rather than all twelve.
    """
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    if stored.get(module) == want:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  {module}: prompt changed since that checkpoint '
              f'({stored.get(module) or "no fingerprint"} -> {want}); '
              f'discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, module: str) -> None:
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    stored[module] = want
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, module: str, input_path=None, output_path=None,
              label='', extra_args=()) -> Path:
    """Run one pipeline stage as a subprocess. Returns the output path.

    CLI flags override profile values (rca run applies a profile key only
    when the matching arg is still at its default), so input/output can be
    redirected for a smoke test without editing the profile.
    """
    spec = json.loads(profile.read_text(encoding='utf-8'))
    resolved_in = Path(input_path) if input_path else BASE / spec['input']
    resolved_out = Path(output_path) if output_path else BASE / spec['output']
    resolved_out.parent.mkdir(parents=True, exist_ok=True)

    # Checkpoints are keyed by the input file's STEM, not its full path, so two
    # notebooks using the same basename share - and silently resume from - each
    # other's checkpoint. runs/checkpoints/smoke_input_checkpoint.* already
    # exists from notebook 06 for a file in a different directory. Namespacing
    # every stage input sd_* makes the collision impossible.
    assert resolved_in.stem.startswith('sd_'), (
        f'{resolved_in.name}: stage inputs must be named sd_* so their checkpoint '
        f'stem cannot collide with another notebook')

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(resolved_in), '--output', str(resolved_out), *map(str, extra_args)]

    banner = label or profile.stem
    print('=' * 78)
    print(f'STAGE: {banner}')
    print('=' * 78)
    _sync_checkpoint(resolved_in, module)
    print(' '.join(str(c) for c in cmd), flush=True)

    err_lines = []

    def _drain(stream):
        for line in iter(stream.readline, ''):
            err_lines.append(line)

    t0 = time.time()
    proc = subprocess.Popen(cmd, text=True, encoding='utf-8', errors='replace',
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            bufsize=1, env=SUBPROCESS_ENV)
    t = threading.Thread(target=_drain, args=(proc.stderr,), daemon=True)
    t.start()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    t.join(timeout=5)

    if proc.returncode != 0:
        print(f'\nExit code {proc.returncode}')
        if err_lines:
            print('--- stderr (last 60 lines) ---')
            print(''.join(err_lines[-60:]))
        raise RuntimeError(f'{banner} failed with exit code {proc.returncode}')

    _stamp_checkpoint(resolved_in, module)
    print(f'\n-> {banner} done in {time.time() - t0:.0f}s  ->  {resolved_out}')
    return resolved_out


print('Checkpoint guard armed. Fingerprints:')
for _m in ALL_MODULES:
    print(f'  {_m:<28} {PROMPT_FINGERPRINTS[_m]}')

## 6b. Gold labels — the eval set and its two strata

`Final List` holds 163 tickets with both a human contact reason and root cause. **61 of them are
in the 2026 run population and 102 are not** (most of the 102 are 2025-dated tickets that can
never appear in a 2026 run). That split is what makes an honest accuracy figure possible:

| | n | scored on | leakage |
|---|---|---|---|
| **A** blind in-run | 61 | the full-run output | clean now, contaminated once feedback is injected |
| **B** held out | 102 | the gold sheet's own text | **structurally clean — this is the headline** |
| **C** feedback fit | 61 | after feedback injection | contaminated, diagnostic only |

**B is the number to report.** Feedback examples are drawn from misclassified *in-run* tickets,
so A stops being blind the moment feedback is added; B never enters the feedback store and stays
comparable before and after, which makes it the only figure that can answer "did feedback help?"

Three label reconciliations are needed, all verified against this workbook:

- `Wrong account / unexpected account switching` → the taxonomy's `Wrong account unexpected
  account switching` (a slash).
- `Not SD related` → `Not for SD`.
- Ticket `260205-018943` has the **contact reason pasted into the root-cause cell**. That is a
  data-entry error, not a label variant, so it is excluded from the root-cause denominator and
  disclosed — counting it as a model error would quietly understate accuracy.

Note the gold sheet's `Conversation` text is **not** the same text as the CSV's: of the 61
overlapping tickets only 1 is byte-identical (gold mean 2,080 chars vs CSV 2,314). They are
separate extracts, so A and B are measuring slightly different things — which is itself worth
knowing if they diverge.

In [ ]:
gold_raw = pd.read_excel(GOLD_XLSX, sheet_name=SHEET_GOLD, dtype=str)
gold_raw.columns = [norm_label(c) for c in gold_raw.columns]

gold = gold_raw[gold_raw['Root cause (select)'].notna()].copy()
gold[ID_COL]     = gold['Unique Identifier Ticket'].map(norm_label)
gold['gold_cr']  = gold['Contact reason (select)'].map(norm_label)
gold['gold_rc']  = gold['Root cause (select)'].map(norm_label)
gold[TEXT_COL]   = gold[TEXT_COL].astype(str)
gold = gold[gold[ID_COL] != ''].drop_duplicates(subset=[ID_COL], keep='first')
print(f'Labelled gold tickets: {len(gold)}')

# ---- Reconcile gold labels onto taxonomy labels ------------------------
GOLD_CR_ALIASES = {}
GOLD_RC_ALIASES = {
    'wrong account / unexpected account switching': 'Wrong account unexpected account switching',
    'not sd related': 'Not for SD',
}
# Contact reason pasted into the root-cause cell: a data error, not a variant.
GOLD_RC_EXCLUDE_IDS = {'260205-018943'}

ALL_CR = set(CR_LABELS.values())
ALL_RC = {rc for rcs in CR_TO_RC.values() for rc in rcs}


def map_gold(value, universe, aliases):
    """Map a human label onto a taxonomy label, or None if it has no equivalent."""
    key = norm_cmp(value)
    if not key:
        return None
    by_norm = {norm_cmp(u): u for u in universe}
    return by_norm.get(key) or aliases.get(key)


gold['gold_cr_mapped'] = gold['gold_cr'].map(lambda v: map_gold(v, ALL_CR, GOLD_CR_ALIASES))
gold['gold_rc_mapped'] = gold['gold_rc'].map(lambda v: map_gold(v, ALL_RC, GOLD_RC_ALIASES))
gold.loc[gold[ID_COL].isin(GOLD_RC_EXCLUDE_IDS), 'gold_rc_mapped'] = None

_bad_cr = gold.loc[gold['gold_cr_mapped'].isna(), 'gold_cr']
_bad_rc = gold.loc[gold['gold_rc_mapped'].isna() & ~gold[ID_COL].isin(GOLD_RC_EXCLUDE_IDS),
                   'gold_rc']
assert _bad_cr.empty, f'gold contact reasons with no taxonomy equivalent: {dict(_bad_cr.value_counts())}'
assert _bad_rc.empty, f'gold root causes with no taxonomy equivalent: {dict(_bad_rc.value_counts())}'
print(f'All {len(gold)} gold contact reasons and '
      f'{int(gold["gold_rc_mapped"].notna().sum())} root causes map onto the taxonomy '
      f'({len(GOLD_RC_EXCLUDE_IDS)} excluded as a data-entry error).')

# ---- Are the gold (CR, RC) pairs actually allowed? --------------------
# pd.notna, not bare truthiness: assigning None into an object column yields NaN,
# and `if nan` is True, so a bare test would report the excluded row as 'nan'.
_violations = [(r['gold_cr_mapped'], r['gold_rc_mapped']) for _, r in gold.iterrows()
               if pd.notna(r['gold_rc_mapped'])
               and r['gold_rc_mapped'] not in CR_TO_RC[r['gold_cr_mapped']]]
if _violations:
    print(f'\n{len(_violations)} gold pair(s) outside the allowed CR/RC mapping:')
    for cr, rc in sorted(set(_violations)):
        print(f'  {rc!r} is not an allowed root cause for {cr!r}')
    print('  (kept for stage-1 scoring; they cannot be predicted by the owning module)')

# ---- Split into the two strata ----------------------------------------
run_ids = set(pd.read_csv(STAGE1_INPUT, dtype=str, usecols=[ID_COL])[ID_COL].str.strip())
rule_ids = set(pd.read_csv(INTER_DIR / 'sd_2026_rule_assigned.csv',
                           dtype=str, usecols=[ID_COL])[ID_COL].str.strip())
population = run_ids | rule_ids
gold['eval_stratum'] = ['in_run' if i in population else 'held_out' for i in gold[ID_COL]]

GOLD_IN_RUN   = gold[gold['eval_stratum'] == 'in_run'].copy()
GOLD_HELD_OUT = gold[gold['eval_stratum'] == 'held_out'].copy()
print(f'\nStratum A (in the run population) : {len(GOLD_IN_RUN):>4}')
print(f'Stratum B (held out, HEADLINE)    : {len(GOLD_HELD_OUT):>4}')

print('\nHeld-out contact reason distribution (the headline denominator):')
print(GOLD_HELD_OUT['gold_cr_mapped'].value_counts().to_string())
_top = GOLD_HELD_OUT['gold_cr_mapped'].value_counts()
print(f'\nMajority-class baseline on stratum B: {_top.iloc[0] / len(GOLD_HELD_OUT) * 100:.1f}% '
      f'({_top.index[0]!r}). Report macro-F1 alongside accuracy or this number flatters '
      f'the model.')

_zero = [cr for cr in CR_ORDER if cr not in set(gold['gold_cr_mapped'])]
print(f'\nContact reasons with NO labelled example: {_zero}')
_unvalidated = [f'sd_rc_{s}' for cr, s, _p, _st in RC_MODULES if cr in _zero]
print(f'Root-cause modules that therefore ship unvalidated: {_unvalidated}')

# Gold text is a different extract from the CSV text.
_csv_text = pd.read_csv(STAGE1_INPUT, dtype=str).set_index(ID_COL)[TEXT_COL]
_cmp = GOLD_IN_RUN[GOLD_IN_RUN[ID_COL].isin(_csv_text.index)]
_same = sum(1 for _, r in _cmp.iterrows() if r[TEXT_COL] == _csv_text[r[ID_COL]])
print(f'\nOf {len(_cmp)} in-run gold tickets, {_same} have text identical to the run input; '
      f'the gold sheet is a separate extract.')

# Smoke/eval input uses the gold sheet's own text.
GOLD_EVAL_CSV = SMOKE_DIR / 'sd_2026_gold_eval_input.csv'
gold[[ID_COL, TEXT_COL]].to_csv(GOLD_EVAL_CSV, index=False)
print(f'Wrote {len(gold)} rows -> {GOLD_EVAL_CSV.relative_to(BASE)}')

## 6c. Scoring helpers

`score_two_stage` reports stage 2 **two ways**, and both matter:

- **Root cause given correct routing** — the module's own quality, measured only on tickets
  stage 1 sent to the right module. Tickets lost to misrouting are counted and printed, never
  silently dropped.
- **End-to-end** — contact reason *and* root cause both correct. This is what ops actually
  experiences.

Reporting only the first flatters the pipeline; only the second hides which stage to fix.

In [ ]:
from sklearn.metrics import classification_report, f1_score


def score_two_stage(pred_cr: pd.DataFrame, pred_rc: pd.DataFrame,
                    gold_df: pd.DataFrame, title: str, *, verbose=True) -> dict:
    """Score one stratum. pred_cr needs [ID_COL,'pred_cr']; pred_rc [ID_COL,'pred_rc']."""
    ev = gold_df[[ID_COL, 'gold_cr_mapped', 'gold_rc_mapped', 'eval_stratum']].copy()
    ev = ev.merge(pred_cr[[ID_COL, 'pred_cr']], on=ID_COL, how='left')
    ev = ev.merge(pred_rc[[ID_COL, 'pred_rc']], on=ID_COL, how='left')
    ev['pred_cr'] = ev['pred_cr'].fillna('')
    ev['pred_rc'] = ev['pred_rc'].fillna('')

    s1 = ev[ev['gold_cr_mapped'].notna() & (ev['pred_cr'] != '')]
    cr_hit = int((s1['gold_cr_mapped'] == s1['pred_cr']).sum())
    cr_acc = cr_hit / len(s1) if len(s1) else float('nan')
    cr_f1 = (f1_score(s1['gold_cr_mapped'], s1['pred_cr'], average='macro', zero_division=0)
             if len(s1) else float('nan'))

    scorable_rc = ev[ev['gold_rc_mapped'].notna() & ev['gold_cr_mapped'].notna()]
    routed = scorable_rc[(scorable_rc['pred_cr'] == scorable_rc['gold_cr_mapped'])
                         & (scorable_rc['pred_rc'] != '')]
    lost = len(scorable_rc) - len(routed)
    rc_hit = int((routed['gold_rc_mapped'] == routed['pred_rc']).sum())
    rc_acc = rc_hit / len(routed) if len(routed) else float('nan')
    rc_f1 = (f1_score(routed['gold_rc_mapped'], routed['pred_rc'], average='macro',
                      zero_division=0) if len(routed) else float('nan'))
    e2e_hit = int(((scorable_rc['gold_cr_mapped'] == scorable_rc['pred_cr'])
                   & (scorable_rc['gold_rc_mapped'] == scorable_rc['pred_rc'])).sum())
    e2e_acc = e2e_hit / len(scorable_rc) if len(scorable_rc) else float('nan')

    if verbose:
        print('=' * 78)
        print(title)
        print('=' * 78)
        print(f'STAGE 1 contact reason : {cr_hit}/{len(s1)} = {cr_acc * 100:.1f}%   '
              f'macro-F1 {cr_f1:.3f}')
        print(f'STAGE 2 root cause     : {rc_hit}/{len(routed)} = {rc_acc * 100:.1f}%   '
              f'macro-F1 {rc_f1:.3f}   (only tickets stage 1 routed correctly)')
        if lost:
            print(f'   {lost} labelled ticket(s) never reached the right root-cause module '
                  f'because stage 1 sent them elsewhere.')
        print(f'END-TO-END both correct: {e2e_hit}/{len(scorable_rc)} = {e2e_acc * 100:.1f}%')

        if len(s1):
            print('\n--- stage 1 by contact reason ---')
            print(classification_report(s1['gold_cr_mapped'], s1['pred_cr'], zero_division=0))
        wrong = s1[s1['gold_cr_mapped'] != s1['pred_cr']]
        if len(wrong):
            print('Stage 1 confusions (gold -> predicted):')
            for (g, p), n in (wrong.groupby(['gold_cr_mapped', 'pred_cr']).size()
                              .sort_values(ascending=False).items()):
                print(f'  {n:>2}x  {g}  ->  {p}')

    return {'title': title, 'n_cr': len(s1), 'cr_acc': cr_acc, 'cr_macro_f1': cr_f1,
            'n_rc_routed': len(routed), 'rc_acc': rc_acc, 'rc_macro_f1': rc_f1,
            'n_rc_scorable': len(scorable_rc), 'e2e_acc': e2e_acc,
            'misrouted': lost, 'eval': ev}


def run_stage2_over(cr_pred: pd.DataFrame, text_src: pd.DataFrame, *, out_dir: Path,
                    tag: str, label: str) -> pd.DataFrame:
    """Route tickets by predicted CR code and run each root-cause module.

    cr_pred needs [ID_COL, 'cr_code']. text_src supplies [ID_COL, TEXT_COL].
    Returns one row per ticket that received a root cause.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    text = text_src[[ID_COL, TEXT_COL]].copy()
    text[ID_COL] = text[ID_COL].astype(str).str.strip()

    sizes = {c: int((cr_pred['cr_code'] == c).sum()) for c in ROUTES}
    order = sorted((c for c in ROUTES if ROUTES[c]['kind'] == 'llm'),
                   key=lambda c: -sizes[c])          # largest subset first
    frames = []
    for code_ in order:
        route, n = ROUTES[code_], sizes[code_]
        if n == 0:
            print(f'  {route["slug"]:<26} empty subset - skipping')
            continue
        ids = set(cr_pred.loc[cr_pred['cr_code'] == code_, ID_COL])
        sub = text[text[ID_COL].isin(ids)].reset_index(drop=True)
        assert len(sub) == n, f'{route["slug"]}: {len(sub)} rows for {n} predicted'
        sub_csv = out_dir / f'sd_2026{tag}_subset_{route["slug"]}.csv'
        sub.to_csv(sub_csv, index=False)

        out = run_stage(route['profile'], module=route['module'], input_path=sub_csv,
                        output_path=out_dir / f'sd_2026{tag}_step2_{route["slug"]}.xlsx',
                        label=f'{label} - {route["slug"]} ({n} tickets)')
        res = read_stage_output(out)
        assert len(res) == n, f'{route["slug"]}: row count changed ({len(res)} != {n})'
        stub, labels = route['stub'], leaf_labels(route['module'])
        # post_process() adds <stub>_evidence_grounded: False when the quoted
        # evidence is not found verbatim in the transcript (flagged, not blanked).
        gcol = f'{stub}_evidence_grounded'
        frames.append(pd.DataFrame({
            ID_COL: res[ID_COL],
            'root_cause_code': res[f'{stub}_code'].astype(str).str.strip(),
            'root_cause': res[f'{stub}_code'].map(lambda c: labels.get(str(c).strip(), '')),
            'root_cause_justification': res[f'{stub}_justification'],
            'root_cause_evidence': res[f'{stub}_evidence'],
            'root_cause_evidence_grounded': (res[gcol] if gcol in res.columns else pd.NA),
            'root_cause_module': route['module'],
            'classification_source': 'llm',
        }))

    # Deterministic contact reasons: exactly one allowed root cause, no LLM call.
    for code_, route in ROUTES.items():
        if route['kind'] != 'deterministic' or sizes[code_] == 0:
            continue
        ids = sorted(cr_pred.loc[cr_pred['cr_code'] == code_, ID_COL])
        print(f'  {route["slug"]:<26} {len(ids):>5} tickets -> '
              f'{route["root_cause"]!r} by rule (no LLM call)')
        frames.append(pd.DataFrame({
            ID_COL: ids,
            'root_cause_code': '',        # no code: it belongs to no module's taxonomy
            'root_cause': route['root_cause'],
            'root_cause_justification': 'Only one root cause is allowed for this contact reason.',
            'root_cause_evidence': '',
            'root_cause_evidence_grounded': pd.NA,
            'root_cause_module': '',
            'classification_source': 'deterministic:single_rc',
        }))

    rc = (pd.concat(frames, ignore_index=True) if frames
          else pd.DataFrame(columns=[ID_COL, 'root_cause_code', 'root_cause',
                                     'root_cause_justification', 'root_cause_evidence',
                                     'root_cause_evidence_grounded', 'root_cause_module',
                                     'classification_source']))
    assert rc[ID_COL].is_unique, 'a ticket received two root causes'
    return rc


print('Scoring and routing helpers defined.')

## 7. Gate 3 — 30-ticket smoke test *(~$0.25)*

Stratified across contact reasons so rare categories are represented. The point is not accuracy
at n=30 — it is to prove the plumbing works end to end: valid JSON, no blank codes, codes that
exist in the taxonomy, and justifications that read like reasoning rather than restatement.

**Read the output.** Everything after this costs real money.

In [ ]:
SMOKE_N = 30
smoke_gold = (gold.groupby('gold_cr_mapped', group_keys=False)
              .apply(lambda g: g.head(max(1, SMOKE_N // gold['gold_cr_mapped'].nunique())),
                     include_groups=False)
              .head(SMOKE_N))
smoke_gold = gold[gold[ID_COL].isin(set(smoke_gold[ID_COL]))].copy()

smoke_in = SMOKE_DIR / 'sd_2026_smoke_input.csv'
smoke_gold[[ID_COL, TEXT_COL]].to_csv(smoke_in, index=False)
print(f'{len(smoke_gold)} tickets across {smoke_gold["gold_cr_mapped"].nunique()} '
      f'contact reasons:')
print(smoke_gold['gold_cr_mapped'].value_counts().to_string())

smoke_out = run_stage(PROFILE_CR, module=MOD_CR, input_path=smoke_in,
                      output_path=SMOKE_DIR / 'smoke_step1.xlsx',
                      label=f'SMOKE stage 1 - {len(smoke_gold)} tickets')
smoke_cr = read_stage_output(smoke_out)
smoke_cr['cr_code'] = smoke_cr[f'{STUB_CR}_code'].astype(str).str.strip()
smoke_cr['pred_cr'] = smoke_cr['cr_code'].map(lambda c: CR_LABELS.get(c, ''))

blank = int((smoke_cr['pred_cr'] == '').sum())
bad = sorted(set(smoke_cr.loc[smoke_cr['pred_cr'] == '', 'cr_code']) - {''})
print(f'\nBlank/invalid contact reason codes: {blank}' + (f'  unknown codes: {bad}' if bad else ''))
assert blank == 0, 'stage 1 produced codes outside the taxonomy - fix before spending more'
print(smoke_cr['pred_cr'].value_counts().to_string())

In [ ]:
# Read a few justifications and evidence quotes: are they reasoning, or restating the label?
for _, r in smoke_cr.head(6).iterrows():
    print(f'--- {r[ID_COL]}  {r["cr_code"]}  {r["pred_cr"]}')
    print(f'    why     : {r[f"{STUB_CR}_justification"]}')
    print(f'    evidence: {str(r[f"{STUB_CR}_evidence"])[:200]}')

In [ ]:
smoke_rc = run_stage2_over(smoke_cr[[ID_COL, 'cr_code']], smoke_gold,
                           out_dir=SMOKE_DIR, tag='_smoke', label='SMOKE stage 2')
print(f'\nRoot causes assigned: {len(smoke_rc)}')
print(smoke_rc['root_cause'].replace('', '(blank)').value_counts().to_string())

_ = score_two_stage(smoke_cr[[ID_COL, 'pred_cr']],
                    smoke_rc.rename(columns={'root_cause': 'pred_rc'})[[ID_COL, 'pred_rc']],
                    smoke_gold, f'SMOKE TEST ({len(smoke_gold)} tickets) vs HUMAN LABELS')

## 8. Gate 4 — eval on all 163 labelled tickets *(~$2)*

This is the "eval first" step, and the decision point. It runs both stages blind over the gold
sheet's own text and reports **A** (61 in-run) and **B** (102 held out) separately.

If **B** is poor, edit the definitions in section 2, re-run sections 2–4, and come back here for
$2 — rather than discovering it after eight hours and $275. The per-module fingerprints mean a
one-module fix re-runs one module.

In [ ]:
gold_cr_out = run_stage(PROFILE_CR, module=MOD_CR, input_path=GOLD_EVAL_CSV,
                        output_path=SMOKE_DIR / 'gold_step1.xlsx',
                        label=f'EVAL stage 1 - {len(gold)} labelled tickets')
gold_cr = read_stage_output(gold_cr_out)
gold_cr['cr_code'] = gold_cr[f'{STUB_CR}_code'].astype(str).str.strip()
gold_cr['pred_cr'] = gold_cr['cr_code'].map(lambda c: CR_LABELS.get(c, ''))
assert int((gold_cr['pred_cr'] == '').sum()) == 0, 'blank contact reason codes in the eval pass'

gold_rc = run_stage2_over(gold_cr[[ID_COL, 'cr_code']], gold,
                          out_dir=SMOKE_DIR, tag='_gold', label='EVAL stage 2')
gold_pred_rc = gold_rc.rename(columns={'root_cause': 'pred_rc'})[[ID_COL, 'pred_rc']]
print(f'\nContact reasons predicted on the gold set:')
print(gold_cr['pred_cr'].value_counts().to_string())

In [ ]:
EVAL_RESULTS = {}
EVAL_RESULTS['B'] = score_two_stage(
    gold_cr[[ID_COL, 'pred_cr']], gold_pred_rc, GOLD_HELD_OUT,
    f'B — HELD OUT ({len(GOLD_HELD_OUT)} tickets)  ** HEADLINE, leakage-free **')

In [ ]:
EVAL_RESULTS['A_blind'] = score_two_stage(
    gold_cr[[ID_COL, 'pred_cr']], gold_pred_rc, GOLD_IN_RUN,
    f'A — IN RUN, BLIND ({len(GOLD_IN_RUN)} tickets)  (scored on gold text, '
    f'pre-feedback so still clean)')

In [ ]:
# Every stage-1 disagreement on the held-out stratum, with the model's own reasoning.
_ev = EVAL_RESULTS['B']['eval']
_wrong = _ev[(_ev['gold_cr_mapped'] != _ev['pred_cr']) & (_ev['pred_cr'] != '')]
print(f'{len(_wrong)} stage-1 disagreements on the held-out stratum\n')
_why = gold_cr.set_index(ID_COL)
for _, r in _wrong.iterrows():
    print(f'--- {r[ID_COL]}')
    print(f'    human : {r["gold_cr_mapped"]}')
    print(f'    model : {r["pred_cr"]}')
    print(f'    why   : {_why.loc[r[ID_COL], f"{STUB_CR}_justification"]}')
    print(f'    quote : {str(_why.loc[r[ID_COL], f"{STUB_CR}_evidence"])[:200]}')

### Decision point

Only continue past here if **B**'s stage-1 accuracy clears the majority-class baseline printed in
section 6b by a comfortable margin, and macro-F1 is not near zero. Otherwise the model is mostly
predicting the dominant category and the remaining ~$275 buys a confident-looking but
uninformative dataset.

In [ ]:
STOP_AFTER_EVAL = False  # cleared 2026-08-03 after reviewing the Gate 4 numbers

_b = EVAL_RESULTS['B']
_baseline = (GOLD_HELD_OUT['gold_cr_mapped'].value_counts().iloc[0] / len(GOLD_HELD_OUT))
print(f'Stage 1 accuracy (B) : {_b["cr_acc"] * 100:.1f}%')
print(f'Majority baseline    : {_baseline * 100:.1f}%')
print(f'Macro-F1             : {_b["cr_macro_f1"]:.3f}')
print(f'Root cause | routed  : {_b["rc_acc"] * 100:.1f}%')
print(f'End to end           : {_b["e2e_acc"] * 100:.1f}%')
if STOP_AFTER_EVAL:
    raise SystemExit('STOP_AFTER_EVAL is True — review the numbers, then set it False '
                     'and re-run from here.')

## 9. Gate 5 — 1,000-record scale probe *(~$8)*

163 hand-picked tickets to 20,805 production tickets is a 128× jump onto text that has been
truncated and rule-filtered. This measures the two things the gold set cannot: **throughput** on
real batches, and the **blank-code rate**.

`--limit` is not a `rca run` flag, so the slice is written as its own input file. It gets
its own checkpoint, which is why it does not contaminate the full run.

In [ ]:
PROBE_N = 1000
probe_in = SMOKE_DIR / 'sd_2026_probe_input.csv'
_stage1_all = pd.read_csv(STAGE1_INPUT, dtype=str)
_stage1_all.sample(n=PROBE_N, random_state=0).to_csv(probe_in, index=False)

_t0 = time.time()
probe_out = run_stage(PROFILE_CR, module=MOD_CR, input_path=probe_in,
                      output_path=SMOKE_DIR / 'probe_step1.xlsx',
                      label=f'PROBE stage 1 - {PROBE_N} production tickets')
_elapsed = time.time() - _t0

probe = read_stage_output(probe_out)
probe['cr_code'] = probe[f'{STUB_CR}_code'].astype(str).str.strip()
probe['pred_cr'] = probe['cr_code'].map(lambda c: CR_LABELS.get(c, ''))
_blank_rate = float((probe['pred_cr'] == '').mean())

print(f'\nThroughput   : {len(probe) / _elapsed:.2f} records/sec')
print(f'Blank codes  : {int((probe["pred_cr"] == "").sum())} / {len(probe)} '
      f'({_blank_rate * 100:.2f}%)')
_n_full = len(_stage1_all)
print(f'Projected stage 1 on {_n_full:,} records: '
      f'{_n_full / (len(probe) / _elapsed) / 3600:.1f} h')
print('\nContact reason distribution on production text:')
print((probe['pred_cr'].replace('', '(blank)').value_counts(normalize=True) * 100)
      .round(1).to_string())

assert _blank_rate <= 0.005, (
    f'blank-code rate {_blank_rate * 100:.2f}% exceeds 0.5% — investigate before the full run')
print('\nBlank-code rate within tolerance.')

## 10. Stage 1 — contact reason for every 2026 ticket *(~$150, ~4 h)*

Runs as a subprocess, so a dropped kernel does not kill it. If it is interrupted, **re-run this
cell**: the checkpoint resumes and only unfinished records are re-paid for. The fingerprint guard
means that resume is safe *unless* the prompt changed, in which case the checkpoint is discarded
rather than silently reused.

In [ ]:
step1_out = run_stage(PROFILE_CR, module=MOD_CR, label='FULL stage 1 - contact reason')

step1 = read_stage_output(step1_out)
step1['cr_code'] = step1[f'{STUB_CR}_code'].astype(str).str.strip()
step1['pred_cr'] = step1['cr_code'].map(lambda c: CR_LABELS.get(c, ''))

stage1_input = pd.read_csv(STAGE1_INPUT, dtype=str)
stage1_input[ID_COL] = stage1_input[ID_COL].astype(str).str.strip()
assert len(step1) == len(stage1_input), (
    f'row count changed: {len(step1)} out, {len(stage1_input)} in')
assert step1[ID_COL].is_unique, 'duplicate ticket ids in the stage-1 output'

print(f'\nRows: {len(step1):,}')
print('\nContact reason distribution:')
for label, n in step1['pred_cr'].replace('', '(blank)').value_counts().items():
    print(f'  {n:>6}  {n / len(step1) * 100:>5.1f}%  {label}')

_blank = step1[step1['pred_cr'] == '']
print(f'\nBlank/unclassified: {len(_blank)} ({len(_blank) / len(step1) * 100:.2f}%)')
assert len(_blank) / len(step1) <= 0.005, 'blank rate above 0.5% - investigate'
if len(_blank):
    _clen = stage1_input.set_index(ID_COL)[TEXT_COL].astype(str).str.len()
    print(_blank[[ID_COL]].assign(chars=_blank[ID_COL].map(_clen))
          .sort_values('chars', ascending=False).head(20).to_string(index=False))

## 11–12. Split by contact reason and run the 11 root-cause modules *(~$120, ~4 h)*

`run_stage2_over` does the routing, keyed on **contact reason code** rather than label. Two labels
contain invisible characters — a zero-width space in `Set up teams or departments` and a
non-breaking space in the root-cause list — so a string comparison can silently produce an empty
subset and a whole category would quietly get no root cause. Codes are ASCII.

Subsets are re-joined from `data/sd_stage1_input_2026.csv`, the **truncated** text, so stage 2
sees exactly what stage 1 saw. Re-joining from the raw CSV would reintroduce the 247K-char record.
Text is never read back from a stage-1 `.xlsx`: openpyxl silently truncates cells at 32,767 chars.

Modules run one at a time, largest subset first — `MODULE_REGISTRY` and `_ENRICH_PREFIXES` are
frozen at import and the cost tracker is a module-level global, so parallel stages would corrupt
each other's accounting. Largest-first means an interruption at hour 3 leaves the big categories
done.

In [ ]:
root_causes = run_stage2_over(step1[[ID_COL, 'cr_code']], stage1_input,
                              out_dir=INTER_DIR, tag='', label='FULL stage 2')

print(f'\nRoot causes assigned: {len(root_causes):,} of {len(step1):,} classified tickets')
print(root_causes['classification_source'].value_counts().to_string())
_rc_blank = int((root_causes['root_cause'] == '').sum())
print(f'Blank root causes: {_rc_blank}')
assert _rc_blank / max(len(root_causes), 1) <= 0.005, 'root-cause blank rate above 0.5%'

# Every LLM root cause must be allowed under the contact reason that routed it.
_chk = root_causes.merge(step1[[ID_COL, 'pred_cr']], on=ID_COL, how='left')
_llm = _chk[_chk['classification_source'] == 'llm']
_invalid = _llm[[rc not in CR_TO_RC.get(cr, []) for rc, cr in
                 zip(_llm['root_cause'], _llm['pred_cr'])]]
assert _invalid.empty, (
    f'{len(_invalid)} root cause(s) outside the allowed mapping — routing is broken:\n'
    f'{_invalid[[ID_COL, "pred_cr", "root_cause"]].head()}')
print('Every root cause is allowed under its contact reason.')

## 13. Merge everything back onto the 23,055 tickets

Three populations reunite here, and `classification_source` keeps them distinguishable:

| source | how the labels were assigned |
|---|---|
| `llm` | both stages by model |
| `deterministic:single_rc` | contact reason by model; root cause forced (only one allowed) |
| `rule:short_conversation` | both by rule — the conversation carried no information |

That column exists because "how many tickets had no usable content?" is a question ops will ask,
and the honest answer is the rule-assigned rows plus those the model itself called
`Not enough info` — *not* tickets where a root-cause module chose its own `Not enough info` leaf
under some other contact reason. Without the column those three are indistinguishable.

In [ ]:
NOT_ENOUGH_INFO = 'Not enough info'

# -- 1. LLM-classified tickets --------------------------------------------
_s1_cols = [ID_COL, 'cr_code', 'pred_cr', f'{STUB_CR}_justification',
            f'{STUB_CR}_evidence']
if f'{STUB_CR}_evidence_grounded' in step1.columns:
    _s1_cols.append(f'{STUB_CR}_evidence_grounded')
merged = step1[_s1_cols].rename(columns={
    'cr_code': 'contact_reason_code', 'pred_cr': 'contact_reason',
    f'{STUB_CR}_justification': 'contact_reason_justification',
    f'{STUB_CR}_evidence': 'contact_reason_evidence',
    f'{STUB_CR}_evidence_grounded': 'contact_reason_evidence_grounded'})
merged = merged.merge(root_causes, on=ID_COL, how='left')
merged['classification_source'] = merged['classification_source'].fillna('llm')

# -- 2. Rule-assigned short tickets ---------------------------------------
rule_rows = pd.read_csv(INTER_DIR / 'sd_2026_rule_assigned.csv', dtype=str)
rule_rows[ID_COL] = rule_rows[ID_COL].astype(str).str.strip()
rule_frame = pd.DataFrame({
    ID_COL: rule_rows[ID_COL],
    'contact_reason_code': LABEL_TO_CR_CODE[NOT_ENOUGH_INFO],
    'contact_reason': NOT_ENOUGH_INFO,
    'contact_reason_justification': (
        f'Conversation payload is under {SHORT_CHAR_THRESHOLD} characters and carries no '
        f'classifiable content, so it was assigned by rule without an LLM call.'),
    'contact_reason_evidence': '',
    'contact_reason_evidence_grounded': pd.NA,
    'root_cause_code': '', 'root_cause': NOT_ENOUGH_INFO,
    'root_cause_justification': 'Assigned by the same rule as the contact reason.',
    'root_cause_evidence': '', 'root_cause_evidence_grounded': pd.NA,
    'root_cause_module': '',
    'classification_source': 'rule:short_conversation',
})

final = pd.concat([merged, rule_frame], ignore_index=True)
for col in ('root_cause_code', 'root_cause', 'root_cause_justification',
            'root_cause_evidence', 'root_cause_module'):
    final[col] = final[col].fillna('')
# The grounding flags stay nullable: NA means "no evidence was quoted", which is
# a different statement from False ("quoted, but not found in the transcript").

# -- 3. Re-attach metadata and the untruncated conversation ---------------
meta = tickets[[ID_COL, 'Incident_ID', 'Channel', DATE_COL, 'parsed_date',
                'conversation_chars_original', TEXT_COL]].copy()
meta[ID_COL] = meta[ID_COL].astype(str).str.strip()
final = meta.merge(final, on=ID_COL, how='left')
final['was_truncated'] = final['conversation_chars_original'] > TRUNC_CAP
final['month'] = final['parsed_date'].dt.to_period('M').astype(str)

# ---- Integrity ----------------------------------------------------------
assert len(final) == len(tickets), f'expected {len(tickets)} rows, got {len(final)}'
assert final[ID_COL].is_unique, 'duplicate ticket ids in the final frame'
assert final['contact_reason'].notna().all(), 'a ticket has no contact reason'
assert final['classification_source'].notna().all(), 'a ticket has no classification source'

_n_llm = int((final['classification_source'] == 'llm').sum())
_n_det = int((final['classification_source'] == 'deterministic:single_rc').sum())
_n_rule = int((final['classification_source'] == 'rule:short_conversation').sum())
assert _n_llm + _n_det + _n_rule == len(final), 'classification_source does not partition'

print(f'Final frame: {len(final):,} rows x {len(final.columns)} columns\n')
print(f'  {_n_llm:>6}  both stages by model')
print(f'  {_n_det:>6}  contact reason by model, root cause forced (single allowed)')
print(f'  {_n_rule:>6}  both by rule (conversation carried no information)')
print(f'\n  with a root cause : {int((final["root_cause"] != "").sum()):,}')
print(f'  truncated for the model: {int(final["was_truncated"].sum())}')

_ni = final['contact_reason'] == NOT_ENOUGH_INFO
print(f'\n"{NOT_ENOUGH_INFO}" as contact reason: {int(_ni.sum()):,} tickets')
print(f'  of which by rule  : {int((_ni & (final["classification_source"] == "rule:short_conversation")).sum()):,}')
print(f'  of which by model : {int((_ni & (final["classification_source"] != "rule:short_conversation")).sum()):,}')
print('\nAll integrity checks passed.')

## 14. Score the full run

**A** is recomputed here against the real run output (rather than the gold sheet's text), which
is the figure that describes the delivered dataset. **B** stays the headline for generalisation
because those 102 tickets are not in the run at all.

In [ ]:
EVAL_RESULTS['A_full_run'] = score_two_stage(
    final[[ID_COL, 'contact_reason']].rename(columns={'contact_reason': 'pred_cr'}),
    final[[ID_COL, 'root_cause']].rename(columns={'root_cause': 'pred_rc'}),
    GOLD_IN_RUN, f'A — IN RUN, scored on the delivered dataset ({len(GOLD_IN_RUN)} tickets)')

In [ ]:
EVAL_SUMMARY = pd.DataFrame([
    {'stratum': 'B (held out)', 'n_contact_reason': EVAL_RESULTS['B']['n_cr'],
     'contact_reason_acc_%': round(EVAL_RESULTS['B']['cr_acc'] * 100, 1),
     'contact_reason_macro_f1': round(EVAL_RESULTS['B']['cr_macro_f1'], 3),
     'n_root_cause_routed': EVAL_RESULTS['B']['n_rc_routed'],
     'root_cause_acc_%': round(EVAL_RESULTS['B']['rc_acc'] * 100, 1),
     'end_to_end_acc_%': round(EVAL_RESULTS['B']['e2e_acc'] * 100, 1),
     'misrouted': EVAL_RESULTS['B']['misrouted'],
     'note': 'HEADLINE. Never used as feedback, so leakage-free and comparable across runs.'},
    {'stratum': 'A (in run, blind)', 'n_contact_reason': EVAL_RESULTS['A_blind']['n_cr'],
     'contact_reason_acc_%': round(EVAL_RESULTS['A_blind']['cr_acc'] * 100, 1),
     'contact_reason_macro_f1': round(EVAL_RESULTS['A_blind']['cr_macro_f1'], 3),
     'n_root_cause_routed': EVAL_RESULTS['A_blind']['n_rc_routed'],
     'root_cause_acc_%': round(EVAL_RESULTS['A_blind']['rc_acc'] * 100, 1),
     'end_to_end_acc_%': round(EVAL_RESULTS['A_blind']['e2e_acc'] * 100, 1),
     'misrouted': EVAL_RESULTS['A_blind']['misrouted'],
     'note': 'Scored on the gold sheet text before any feedback. Becomes contaminated '
             'once these tickets seed the feedback store.'},
    {'stratum': 'A (in run, delivered)', 'n_contact_reason': EVAL_RESULTS['A_full_run']['n_cr'],
     'contact_reason_acc_%': round(EVAL_RESULTS['A_full_run']['cr_acc'] * 100, 1),
     'contact_reason_macro_f1': round(EVAL_RESULTS['A_full_run']['cr_macro_f1'], 3),
     'n_root_cause_routed': EVAL_RESULTS['A_full_run']['n_rc_routed'],
     'root_cause_acc_%': round(EVAL_RESULTS['A_full_run']['rc_acc'] * 100, 1),
     'end_to_end_acc_%': round(EVAL_RESULTS['A_full_run']['e2e_acc'] * 100, 1),
     'misrouted': EVAL_RESULTS['A_full_run']['misrouted'],
     'note': 'The same tickets as scored in the delivered dataset. Differs from A (blind) '
             'only because the gold sheet and the ticket export are different extracts.'},
])
print(EVAL_SUMMARY.drop(columns=['note']).to_string(index=False))

## 15. Seed the feedback store *(optional iteration)*

Turns the misclassified **in-run** tickets into few-shot corrections, leaving **B** untouched so
it stays a valid before/after measurement.

Each correction is written to its owning module's file and its code is validated against that
module's own taxonomy first. A correction naming a code from another module teaches the model an
invalid code — exactly the trap that made notebook 06 disable feedback for LeapSpace, where 115
corrections carried codes from a retired taxonomy.

After running this, add `feedback_dir` / `feedback_strategy` / `feedback_max_examples` to the
profiles, re-run section 8, and compare **B**. Check the log does *not* warn about falling back
from `smart` to `recent`.

In [ ]:
from rca_pipeline.enhancements.feedback import FeedbackStore

_ev = EVAL_RESULTS['A_full_run']['eval']
_cr_wrong = _ev[(_ev['gold_cr_mapped'].notna()) & (_ev['pred_cr'] != '')
                & (_ev['gold_cr_mapped'] != _ev['pred_cr'])]
_text = final.set_index(ID_COL)[TEXT_COL]

FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)
store = FeedbackStore(FEEDBACK_DIR)
_cr_leaves = set(CR_LABELS)
written = 0
for _, r in _cr_wrong.iterrows():
    want_code = LABEL_TO_CR_CODE.get(r['gold_cr_mapped'])
    if want_code not in _cr_leaves:          # never teach a code the module cannot emit
        print(f'  skipped {r[ID_COL]}: {r["gold_cr_mapped"]!r} has no code in {MOD_CR}')
        continue
    store.add_correction(
        uid=str(r[ID_COL]), module=MOD_CR,
        text=str(_text.get(r[ID_COL], ''))[:4000],
        original={f'{STUB_CR}_code': LABEL_TO_CR_CODE.get(r['pred_cr'], '')},
        corrected={f'{STUB_CR}_code': want_code},
        reason=f'Human label is {r["gold_cr_mapped"]}, not {r["pred_cr"]}.')
    written += 1

print(f'{written} contact-reason correction(s) -> '
      f'{(FEEDBACK_DIR / f"{MOD_CR}_corrections.jsonl").relative_to(BASE)}')

# Root-cause corrections, only where stage 1 routed the ticket correctly (otherwise
# the correction belongs to a module that never saw the ticket).
_routed = _ev[(_ev['gold_cr_mapped'] == _ev['pred_cr']) & _ev['gold_rc_mapped'].notna()
              & (_ev['pred_rc'] != '') & (_ev['gold_rc_mapped'] != _ev['pred_rc'])]
_rc_written = {}
for _, r in _routed.iterrows():
    route = ROUTES.get(LABEL_TO_CR_CODE[r['gold_cr_mapped']])
    if not route or route['kind'] != 'llm':
        continue
    labels = leaf_labels(route['module'])
    code_of = {v: k for k, v in labels.items()}
    want = code_of.get(r['gold_rc_mapped'])
    if want is None:
        print(f'  skipped {r[ID_COL]}: {r["gold_rc_mapped"]!r} is not a leaf of '
              f'{route["module"]}')
        continue
    store.add_correction(
        uid=str(r[ID_COL]), module=route['module'],
        text=str(_text.get(r[ID_COL], ''))[:4000],
        original={f'{route["stub"]}_code': code_of.get(r['pred_rc'], '')},
        corrected={f'{route["stub"]}_code': want},
        reason=f'Human label is {r["gold_rc_mapped"]}, not {r["pred_rc"]}.')
    _rc_written[route['module']] = _rc_written.get(route['module'], 0) + 1

for m, n in sorted(_rc_written.items()):
    print(f'{n} root-cause correction(s) -> {m}_corrections.jsonl')
print(f'\nTotal: {written + sum(_rc_written.values())} corrections across '
      f'{1 + len(_rc_written)} module file(s).')
print('\nTo use them, add to each profile and re-run section 8:')
print('  "feedback_dir": "feedback/sd_2026", "feedback_strategy": "smart", '
      '"feedback_max_examples": 5')

## 16. Write the deliverable

Nine sheets. `fit_excel` truncates over-long cells **visibly** rather than letting openpyxl do it
silently at 32,767 chars, and the companion CSV carries the untruncated text.

The per-row `*_description` columns are dropped — they would repeat the whole taxonomy on every
one of 23,055 rows — and the taxonomy appears once in its own sheet instead.

`Eval methodology` is not optional. Anyone reading only the workbook must be able to see that two
root-cause modules were never validated, and that the headline is measured on 102 tickets.

In [ ]:
XL_LIMIT = 32_767
MARKER = '  ...[TRUNCATED FOR EXCEL - see companion CSV for full text]'

OUT_COLS = [ID_COL, 'Incident_ID', 'Channel', DATE_COL, 'month',
            'contact_reason_code', 'contact_reason', 'contact_reason_justification',
            'contact_reason_evidence', 'contact_reason_evidence_grounded',
            'root_cause_code', 'root_cause', 'root_cause_justification',
            'root_cause_evidence', 'root_cause_evidence_grounded',
            'root_cause_module', 'classification_source',
            'conversation_chars_original', 'was_truncated', TEXT_COL]
deliverable = final[OUT_COLS].copy()
deliverable.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    long = df[TEXT_COL].astype(str).str.len() > XL_LIMIT
    df.loc[long, TEXT_COL] = (df.loc[long, TEXT_COL].astype(str)
                              .str[:XL_LIMIT - len(MARKER)] + MARKER)
    return df


def summary(df: pd.DataFrame, col: str, denom: int) -> pd.DataFrame:
    vc = df[col].replace('', '(blank)').value_counts()
    out = pd.DataFrame({col: vc.index, 'tickets': vc.values,
                        'share_%': (vc.values / max(denom, 1) * 100).round(1)})
    return out


# -- contact reason summary, with a complexity proxy and a taxonomy-health column
cr_sum = summary(deliverable, 'contact_reason', len(deliverable))
_chars = deliverable.groupby('contact_reason')['conversation_chars_original'].mean().round(0)
cr_sum['mean_conversation_chars'] = cr_sum['contact_reason'].map(_chars)
_catchall = (deliverable.assign(_c=deliverable['root_cause'].isin(['Other', NOT_ENOUGH_INFO]))
             .groupby('contact_reason')['_c'].mean() * 100).round(1)
cr_sum['other_or_no_info_%'] = cr_sum['contact_reason'].map(_catchall)

# -- root cause summary: share within contact reason AND of the whole population
rc_rows = []
for cr in cr_sum['contact_reason']:
    sub = deliverable[deliverable['contact_reason'] == cr]
    if sub.empty:
        continue
    s = summary(sub, 'root_cause', len(sub)).rename(columns={'share_%': 'share_within_cr_%'})
    s.insert(0, 'contact_reason', cr)
    s['share_of_total_%'] = (s['tickets'] / len(deliverable) * 100).round(2)
    rc_rows.append(s)
rc_sum = pd.concat(rc_rows, ignore_index=True)

cr_by_rc = pd.crosstab(deliverable['contact_reason'],
                       deliverable['root_cause'].replace('', '(none)'))
cr_by_month = pd.crosstab(deliverable['contact_reason'], deliverable['month'])
cr_by_channel = pd.crosstab(deliverable['contact_reason'], deliverable['Channel'])
top_rc = (summary(deliverable[deliverable['root_cause'] != ''], 'root_cause',
                  len(deliverable)).head(20))

eval_rows = EVAL_RESULTS['B']['eval'].copy()
for k in ('A_blind', 'A_full_run'):
    eval_rows = pd.concat([eval_rows, EVAL_RESULTS[k]['eval']], ignore_index=True)
eval_rows = eval_rows.drop_duplicates(subset=[ID_COL, 'eval_stratum'], keep='last')

tax_rows = [{'module': m, 'contact_reason_group': node.get('label', ''), 'code': c,
             'label': leaf.get('label', ''),
             'guidance': str(leaf.get('description', ''))[:XL_LIMIT]}
            for m in ALL_MODULES
            for node in json.loads((TAXONOMY_DIR / f'{m}.json').read_text(encoding='utf-8')).values()
            for c, leaf in node.get('children', {}).items()]

_zero_cr = [cr for cr in CR_ORDER if cr not in set(gold['gold_cr_mapped'])]
_unval = [f'sd_rc_{s}' for cr, s, _p, _st in RC_MODULES if cr in _zero_cr]
methodology = pd.DataFrame([
    ('Ticket population', f'{len(deliverable):,} ScienceDirect tickets dated in calendar '
                          f'{TARGET_YEAR}, from {TICKETS_CSV.name}.'),
    ('Excluded', f'{len(excluded)} rows dated before {TARGET_YEAR} '
                 f'(see runs/intermediate/sd_2026_excluded_pre2026.csv).'),
    ('Rule-assigned', f'{_n_rule:,} tickets whose conversation payload is under '
                      f'{SHORT_CHAR_THRESHOLD} characters were assigned "{NOT_ENOUGH_INFO}" '
                      f'without an LLM call. See classification_source.'),
    ('Truncation', f'{int(final["was_truncated"].sum())} conversations longer than '
                   f'{TRUNC_CAP:,} chars were cut to the first {TRUNC_HEAD:,} plus the last '
                   f'{TRUNC_TAIL:,}, with a visible marker. ~93% of all text retained. '
                   f'The full text is in the companion CSV.'),
    ('Two-stage design', 'Stage 1 assigns one of 13 contact reasons. Tickets are then routed '
                         'by contact reason CODE to a module holding only that contact '
                         "reason's allowed root causes, so an invalid CR/RC pair cannot "
                         'occur. 2 contact reasons have a single allowed root cause and are '
                         'assigned without an LLM call.'),
    ('HEADLINE accuracy', f'Stratum B: {len(GOLD_HELD_OUT)} human-labelled tickets that are '
                          f'NOT in the run population. Contact reason '
                          f'{EVAL_RESULTS["B"]["cr_acc"] * 100:.1f}% '
                          f'(macro-F1 {EVAL_RESULTS["B"]["cr_macro_f1"]:.3f}); root cause '
                          f'{EVAL_RESULTS["B"]["rc_acc"] * 100:.1f}% on correctly routed '
                          f'tickets; end-to-end {EVAL_RESULTS["B"]["e2e_acc"] * 100:.1f}%.'),
    ('Why B is the headline', 'Feedback examples are drawn from misclassified in-run tickets, '
                              'so stratum A stops being blind once feedback is injected. '
                              'B never enters the feedback store and stays comparable.'),
    ('Confidence interval', f'With n={len(GOLD_HELD_OUT)} the 95% CI on an accuracy near 80% '
                            f'is roughly +/-8 percentage points. Treat differences smaller '
                            f'than that as noise.'),
    ('Majority-class baseline', f'{GOLD_HELD_OUT["gold_cr_mapped"].value_counts().iloc[0] / len(GOLD_HELD_OUT) * 100:.1f}% '
                                f'of stratum B is "{GOLD_HELD_OUT["gold_cr_mapped"].value_counts().index[0]}". '
                                f'Always read macro-F1 next to accuracy.'),
    ('UNVALIDATED modules', f'{len(_zero_cr)} contact reasons have no labelled example '
                            f'({", ".join(_zero_cr)}), so these root-cause modules were never '
                            f'measured: {", ".join(_unval) or "none"}.'),
    ('Excluded gold row', f'Ticket {", ".join(sorted(GOLD_RC_EXCLUDE_IDS))} has the contact '
                          f'reason pasted into the root-cause cell. Excluded from the '
                          f'root-cause denominator as a data-entry error rather than counted '
                          f'as a model error.'),
    ('Evidence grounding', 'Each row carries *_evidence_grounded. False means the quoted '
                           'evidence was not found verbatim in the transcript; NA means no '
                           'quote was offered. Measured at 1.2% False on a 1,000-ticket probe. '
                           'The code and justification are unaffected -- filter on this column '
                           'if you need quotes you can rely on.'),
    ('Gold text differs', f'Of {len(_cmp)} in-run labelled tickets, only {_same} has text '
                          f'identical to the ticket export. The gold workbook is a separate '
                          f'extract, which is why A (blind) and A (delivered) can differ.'),
    ('Taxonomy conflict', 'Def RC files "Request for information" under "Accessing content"; '
                          'Lists CR and RC files it under "Other - not access related". The '
                          'human-labelled data agrees with Lists CR and RC, which was '
                          'therefore used. Worth confirming with the taxonomy owner.'),
    ('Model', f'{json.loads(PROFILE_CR.read_text(encoding="utf-8")).get("model", "azure/dsaic-llm-general (default)")}, '
              f'temperature 0.'),
], columns=['topic', 'detail'])

manifest = pd.DataFrame(
    [('source_file', TICKETS_CSV.name), ('rows_in_source', f'{len(raw):,}'),
     ('rows_in_scope', f'{len(deliverable):,}'), ('rows_excluded_pre_2026', str(len(excluded))),
     ('rows_rule_assigned', f'{_n_rule:,}'), ('rows_deterministic_rc', f'{_n_det:,}'),
     ('rows_llm_both_stages', f'{_n_llm:,}'),
     ('conversations_truncated', str(int(final['was_truncated'].sum()))),
     ('short_char_threshold', str(SHORT_CHAR_THRESHOLD)),
     ('truncation_cap_head_tail', f'{TRUNC_CAP}/{TRUNC_HEAD}/{TRUNC_TAIL}'),
     ('modules', str(len(ALL_MODULES))), ('taxonomy_leaves', str(len(tax_rows))),
     ('definition_workbook', DEFN_XLSX.name),
     ('generated_at', pd.Timestamp.now().isoformat(timespec='seconds'))]
    + [(f'fingerprint_{m}', PROMPT_FINGERPRINTS[m]) for m in ALL_MODULES],
    columns=['key', 'value'])

cost_rows = []
for p in sorted((BASE / 'runs' / 'cost_reports').glob('costs_*.json')):
    try:
        c = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    cost_rows.append({'report': p.name, 'total_cost_usd': c.get('total_cost_usd'),
                      'total_tokens': c.get('total_tokens'), 'calls': c.get('total_calls')})
costs = pd.DataFrame(cost_rows)

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as w:
    fit_excel(deliverable).to_excel(w, sheet_name='All tickets', index=False)
    cr_sum.to_excel(w, sheet_name='Contact reason summary', index=False)
    rc_sum.to_excel(w, sheet_name='Root cause summary', index=False)
    cr_by_rc.to_excel(w, sheet_name='CR x RC matrix')
    cr_by_month.to_excel(w, sheet_name='CR by month')
    cr_by_channel.to_excel(w, sheet_name='CR by channel')
    top_rc.to_excel(w, sheet_name='Top 20 root causes', index=False)
    eval_rows.to_excel(w, sheet_name='Eval vs human labels', index=False)
    EVAL_SUMMARY.to_excel(w, sheet_name='Eval summary', index=False)
    methodology.to_excel(w, sheet_name='Eval methodology', index=False)
    pd.DataFrame(tax_rows).to_excel(w, sheet_name='Taxonomy used', index=False)
    manifest.to_excel(w, sheet_name='Run manifest', index=False)
    if not costs.empty:
        costs.to_excel(w, sheet_name='Cost reports', index=False)

print(f'Workbook : {FINAL_XLSX}')
print(f'Full text: {FINAL_CSV}')
print(f'\nRows: {len(deliverable):,}   Sheets: 12')
print(f'\nHEADLINE (stratum B, n={len(GOLD_HELD_OUT)}): contact reason '
      f'{EVAL_RESULTS["B"]["cr_acc"] * 100:.1f}%, root cause '
      f'{EVAL_RESULTS["B"]["rc_acc"] * 100:.1f}%, end-to-end '
      f'{EVAL_RESULTS["B"]["e2e_acc"] * 100:.1f}%')

## Afterwards — restoring the LeapSpace modules

`generated/pipeline_config.json` now describes the twelve ScienceDirect modules, so the LeapSpace
ones are no longer loadable. Their outputs are untouched under `runs/output/leapspace_2026_*`.
To put them back:

```
rca build --input definitions/leapspace_2026_pipeline_definition.xlsx --force-all
```

That drops the ScienceDirect modules in turn — the config holds one product at a time. The
backup taken in section 0c is in `generated/_backup_pre_sd2026/`, alongside a `RESTORE.txt`.

### Re-running after a taxonomy edit

Edit section 2, then re-run sections 2, 3, 4 and 6. The fingerprint check in section 6 will
discard only the checkpoints of modules whose prompt actually changed, so a one-module fix costs
one module's worth of re-classification rather than the whole run.